# NB05 — Evaluation, statistics and paper figures

**Project:** CardioMamba-Net · **Stage:** 5 of 5
`01_verify` → `02_preprocess` → `03_baselines` → `04_cardiomamba_train` → **`05_evaluate`**

---

## What this produces

Everything the manuscript needs, generated from the per-window metrics that NB03 and NB04 already
wrote — so **no model is retrained here**. Only the robustness section (§7) runs inference, and it
is optional.

NB04 run IDs beginning with `quick__` are validation smoke tests and are excluded by default.
The notebook writes `input_audit.json` and labels its final output **PARTIAL** until the canonical
NB04 queue is complete; it never silently presents one quick fold as a final model result.

| Output | Corresponds to |
|---|---|
| `table2_per_scenario.csv` | Their Table 2 — per-scenario, all models |
| `table3_rva_combined.csv` | Their Table 3 — the headline comparison |
| `table3b/3c/3d_*.csv` | All-five, LOSO, and held-out-scenario generalisation |
| `table4_peak_detection.csv` | Their Table 4 — R-peak accuracy/precision/recall/F1 |
| `table5_hrv.csv` | Their Table 5 — μRR, σRR, μHR, σHR, RMSSD, **in real milliseconds** |
| `table6_ablation.csv` | Ours — the ablation ladder |
| `table7_significance.csv` | Ours — Wilcoxon signed-rank with Holm correction |
| `table8_budget.csv` | Ours — parameters, and correlation per million parameters |
| 10 figures | Bland–Altman ×2, per-subject box plots, qualitative grid, ablation, robustness, budget scatter |

## What the statistics are for

The baseline reports means and standard deviations and stops. That is not enough to claim a win.
Here the full model is compared with every ablation using a **subject-paired Wilcoxon signed-rank
test**, and the p-values are **Holm-corrected**. Five fold averages are too few for a two-sided
Wilcoxon test to reach 0.05 even when every fold moves in the same direction; subjects are the
independent experimental units and provide the defensible paired analysis.

Bland–Altman with limits of agreement is the standard way to report agreement between two
measurement methods in clinical work, and it is what a reviewer from a medical journal will look
for on heart rate and HRV. A correlation coefficient alone does not tell them whether the method
can be trusted on an individual patient.

## ⚠️ Accelerator: **GPU T4 × 2** *(only needed for §7)*

Sections 1–6 and 8 run fine on CPU. If you only want the tables and figures, set
`CFG["RUN_ROBUSTNESS"] = False` and use **Accelerator: None**.

When robustness is enabled, please attach NB02's saved output with **+ Add Input → Notebook
Output**. Tables use the HF run repositories; the attached corpus makes inference faster and
keeps it outside the 20 GB working area.

## Cell-by-cell run guide

| Code cell | What runs | Typical time |
|---:|---|---:|
| 1 | Configuration | < 5 s |
| 2 | Imports/dependencies/output folders | 1–3 min |
| 3 | Write libraries, HF login, start results-repo sync | 1–3 min |
| 4 | Download summaries/metrics from baseline and model repos | 2–15 min |
| 5 | Build waveform comparison tables | < 1 min |
| 6 | Build peak, HR and HRV tables | < 2 min |
| 7 | Ablation, subject-paired Wilcoxon-Holm, compute budget | 1–5 min |
| 8 | Generate manuscript figures | 2–10 min |
| 9 | Optional GPU robustness inference | 20–90 min |
| 10 | Write manuscript summary and final HF upload | 2–15 min |

Total without robustness: **10–45 minutes**. With robustness: typically **30–120 minutes**, mainly
depending on whether inputs/checkpoints are already cached.

---
# 1 · Configuration

In [1]:
CFG = {
    "DATA_REPO":     "Shanmuk4622/cr-rvs-radar-ecg-processed-v2",
    "BASELINE_REPO": "Shanmuk4622/cardiomamba-baselines-v2",
    "MODEL_REPO":    "Shanmuk4622/cardiomamba-net-v2",
    "RESULT_REPO":   "Shanmuk4622/cardiomamba-results-v2",
    "HF_PRIVATE": False,
    "RUN_ID":     "nb05_evaluation_v2",

    "WORK":    "/kaggle/working/nb05",
    "SCRATCH": "/kaggle/temp/nb05",
    "PUSH_INTERVAL_S": 30 * 60,
    "HF_MAX_UPLOADS_HOUR": 24,
    # Remove artifacts restored from an older/partial evaluation before rebuilding them.
    # The final forced upload replaces the canonical HF paths with this run's outputs.
    "REBUILD_CANONICAL_RESULTS": True,

    "RUN_ROBUSTNESS": True,          # needs GPU + checkpoints; set False for tables only
    # Quick NB04 runs are architecture smoke tests, not paper results. Keeping them out by
    # default prevents a single 10-epoch fold being presented as the trained full model.
    "INCLUDE_QUICK_RUNS": False,
    "EXPECTED_BASELINE_RUNS": 80,
    "EXPECTED_NB04_FULL_RUNS": 100,
    "SNR_DB": [12, 6, 3, 0, -3],
    "MOTION_AMPLITUDE": [0.25, 0.50],   # normalised slow-drift amplitudes
    "TEST_CHANNEL_DROPOUT": True,
    "ROBUST_MODELS": ["L9_full", "multireslinknet"],
    "ROBUST_MAX_WINDOWS": 800,

    "HEADLINE_EXP": "B_rva",
    "ALPHA": 0.05,
    "SEED": 1337,
}
import json
print(json.dumps(CFG, indent=2))

{
  "DATA_REPO": "Shanmuk4622/cr-rvs-radar-ecg-processed-v2",
  "BASELINE_REPO": "Shanmuk4622/cardiomamba-baselines-v2",
  "MODEL_REPO": "Shanmuk4622/cardiomamba-net-v2",
  "RESULT_REPO": "Shanmuk4622/cardiomamba-results-v2",
  "HF_PRIVATE": false,
  "RUN_ID": "nb05_evaluation_v2",
  "WORK": "/kaggle/working/nb05",
  "SCRATCH": "/kaggle/temp/nb05",
  "PUSH_INTERVAL_S": 1800,
  "HF_MAX_UPLOADS_HOUR": 24,
  "REBUILD_CANONICAL_RESULTS": true,
  "RUN_ROBUSTNESS": true,
  "INCLUDE_QUICK_RUNS": false,
  "EXPECTED_BASELINE_RUNS": 80,
  "EXPECTED_NB04_FULL_RUNS": 100,
  "SNR_DB": [
    12,
    6,
    3,
    0,
    -3
  ],
  "MOTION_AMPLITUDE": [
    0.25,
    0.5
  ],
  "TEST_CHANNEL_DROPOUT": true,
  "ROBUST_MODELS": [
    "L9_full",
    "multireslinknet"
  ],
  "ROBUST_MAX_WINDOWS": 800,
  "HEADLINE_EXP": "B_rva",
  "ALPHA": 0.05,
  "SEED": 1337
}


In [2]:
import os, sys, gc, json, math, time, warnings, subprocess, itertools, hashlib
from pathlib import Path
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

def _pip(*p):
    import importlib.util
    miss = [x for x in p if importlib.util.find_spec(x.replace("-", "_")) is None]
    if miss:
        print("installing:", miss)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *miss], check=True)
        for x in miss: __import__(x.replace("-", "_"))
_pip("pyarrow", "huggingface_hub")

import numpy as np, pandas as pd
WORK = Path(CFG["WORK"]); SCRATCH = Path(CFG["SCRATCH"])
for d in (WORK, SCRATCH, WORK / "tables", WORK / "figures"):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK))
pd.set_option("display.width", 240, "display.max_columns", 60)
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.3.3 | numpy 2.0.2


In [3]:
MODULES = {
 "crvs_sync.py":    r"""
# crvs_sync.py -- conservative, resumable and interrupt-safe Hugging Face sync.
# A folder upload can involve several HTTP requests, so this deliberately schedules far
# fewer than the nominal API limit: one periodic upload per 30 minutes, plus major stages
# and a best-effort final upload on interrupt. All local writes are atomic.
import os, json, time, random, threading, atexit, signal
from collections import deque
from pathlib import Path
from datetime import datetime, timezone

SYNC_VERSION = 2

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2, default=str)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

class RollingLimiter:
    # Limits upload_folder CALLS, not HTTP requests. The low default leaves a wide margin.
    def __init__(self, calls_per_hour=24, min_gap_s=20):
        self.limit = max(1, int(calls_per_hour)); self.min_gap = float(min_gap_s)
        self.times = deque(); self.lock = threading.Lock()
    def take(self, timeout=1800):
        deadline = time.monotonic() + timeout
        while True:
            with self.lock:
                now = time.monotonic()
                while self.times and now - self.times[0] >= 3600:
                    self.times.popleft()
                gap = self.min_gap - (now - self.times[-1]) if self.times else 0.0
                window = 3600 - (now - self.times[0]) if len(self.times) >= self.limit else 0.0
                wait = max(0.0, gap, window)
                if wait <= 0:
                    self.times.append(now); return True
            if time.monotonic() + wait > deadline:
                return False
            time.sleep(min(wait, 10.0))

class HFSync:
    def __init__(self, repo_id, local_dir, token, repo_type="dataset", private=False,
                 run_id="run", push_interval_s=1800, max_upload_calls_hour=24,
                 retry_max=6, verbose=True):
        from huggingface_hub import HfApi
        if not token:
            raise RuntimeError("HF_TOKEN is missing. Add it under Kaggle > Add-ons > Secrets.")
        self.api = HfApi(token=token); self.token = token
        self.repo_id = repo_id; self.repo_type = repo_type; self.private = private
        self.run_id = run_id
        self.local = Path(local_dir); self.local.mkdir(parents=True, exist_ok=True)
        self.interval = max(300, int(push_interval_s))
        self.limiter = RollingLimiter(max_upload_calls_hour)
        self.retry_max = retry_max; self.verbose = verbose
        self._last_push = time.time(); self._dirty = threading.Event()
        self._force = threading.Event(); self._wake = threading.Event()
        self._stop = threading.Event(); self._upload_lock = threading.Lock()
        self._log_lock = threading.Lock(); self._dirty_lock = threading.Lock()
        self._dirty_generation = 0; self._before_final = None
        self._pushes = 0; self._failures = 0; self._closed = False
        self.history = self.local / "sync_history.jsonl"
        self.state_path = self.local / "sync_state.json"
        self._ensure_repo(); self._install_handlers()
        self._thread = threading.Thread(target=self._loop, daemon=True, name="hf-uploader")
        self._thread.start()
        self.log("sync_started", repo=self.repo_id, private=self.private,
                 interval_s=self.interval, sync_version=SYNC_VERSION)

    def _ensure_repo(self):
        from huggingface_hub import create_repo
        create_repo(self.repo_id, repo_type=self.repo_type, private=self.private,
                    exist_ok=True, token=self.token)

    @property
    def url(self):
        kind = "datasets/" if self.repo_type == "dataset" else ""
        return "https://huggingface.co/" + kind + self.repo_id

    def recently_pushed(self, seconds=10):
        return self._pushes > 0 and (time.time() - self._last_push) <= float(seconds)

    def log(self, event, _mark_dirty=True, **kw):
        rec = {"ts": datetime.now(timezone.utc).isoformat(), "run": self.run_id, "event": event}
        rec.update(kw)
        try:
            with self._log_lock, open(self.history, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, default=str) + "\n"); f.flush()
        except Exception:
            pass
        if _mark_dirty:
            self._touch()
        if self.verbose and event not in ("heartbeat",):
            print("  [" + event + "] " + " ".join(f"{k}={v}" for k, v in kw.items()))

    def _touch(self):
        with self._dirty_lock:
            self._dirty_generation += 1; self._dirty.set()

    def mark_dirty(self, reason=None):
        if reason:
            self.log("dirty", reason=reason)
        else:
            self._touch()

    def save_state(self, state):
        atomic_json(self.state_path, state); self._touch()

    def load_state(self, default=None):
        if self.state_path.exists():
            try:
                return json.loads(self.state_path.read_text(encoding="utf-8"))
            except Exception as e:
                self.log("state_read_error", err=type(e).__name__)
        return default if default is not None else {}

    def pull(self, allow_patterns=None, into=None):
        # Download into the real working folder. Call before producing new local files.
        from huggingface_hub import snapshot_download
        target = Path(into or self.local); target.mkdir(parents=True, exist_ok=True)
        try:
            p = snapshot_download(self.repo_id, repo_type=self.repo_type, token=self.token,
                                  local_dir=str(target), allow_patterns=allow_patterns,
                                  max_workers=4)
            self.log("resume_pull_ok", _mark_dirty=False, path=str(p), patterns=allow_patterns)
            return True
        except Exception as e:
            self.log("resume_pull_empty", _mark_dirty=False,
                     err=f"{type(e).__name__}: {str(e)[:240]}")
            return False

    def set_before_final_flush(self, callback):
        # Trainer registers an atomic emergency-checkpoint callback while it is active.
        self._before_final = callback

    def stage_done(self, name, **kw):
        self.log("stage_done", stage=name, **kw)
        self._force.set(); self._wake.set()

    def _run_final_hook(self):
        cb = self._before_final
        if cb is not None:
            try:
                cb()
            except Exception as e:
                self.log("final_checkpoint_error", err=f"{type(e).__name__}: {e}")

    def _do_upload(self, msg):
        from huggingface_hub import upload_folder
        for attempt in range(self.retry_max):
            if not self.limiter.take(timeout=1800):
                self.log("upload_call_limit_timeout"); return False
            try:
                info = upload_folder(folder_path=str(self.local), repo_id=self.repo_id,
                                     repo_type=self.repo_type, token=self.token,
                                     commit_message=msg,
                                     ignore_patterns=["*.tmp", "**/__pycache__/**", ".git*",
                                                      "*.lock", ".cache/**"])
                self._pushes += 1; self._last_push = time.time()
                meta = {"last_push_utc": datetime.now(timezone.utc).isoformat(),
                        "pushes_this_session": self._pushes,
                        "last_commit": str(getattr(info, "oid", "")), "message": msg}
                atomic_json(self.local / "last_push.json", meta)
                self.log("push_ok", _mark_dirty=False, n=self._pushes,
                         commit=meta["last_commit"], msg=msg)
                return True
            except Exception as e:
                self._failures += 1
                wait = min(300, (2 ** attempt) * 5) * (0.7 + 0.6 * random.random())
                self.log("push_retry", attempt=attempt + 1,
                         err=f"{type(e).__name__}: {str(e)[:500]}", sleep=round(wait, 1))
                time.sleep(wait)
        self.log("push_failed_permanently", msg=msg); return False

    def flush(self, final=False, msg=None, force=False, run_final_hook=False):
        if run_final_hook:
            self._run_final_hook()
        if not self._dirty.is_set() and not force:
            return True
        with self._upload_lock:
            if not self._dirty.is_set() and not force:
                return True
            with self._dirty_lock:
                generation = self._dirty_generation
            stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
            label = "final" if final else "checkpoint"
            message = msg or f"{self.run_id} {label} @ {stamp}Z"
            ok = self._do_upload(message)
            if ok:
                self._force.clear()
                with self._dirty_lock:
                    if self._dirty_generation == generation:
                        self._dirty.clear()
            return ok

    def _loop(self):
        while not self._stop.is_set():
            remaining = max(1.0, self.interval - (time.time() - self._last_push))
            self._wake.wait(min(30.0, remaining)); self._wake.clear()
            if self._stop.is_set():
                break
            forced = self._force.is_set()
            due = (time.time() - self._last_push) >= self.interval
            if self._dirty.is_set() and (due or forced):
                try:
                    tag = "major-stage" if forced else "periodic-30min"
                    self.flush(msg=f"{self.run_id} {tag} @ " +
                               datetime.now(timezone.utc).strftime("%H:%M") + "Z")
                except Exception as e:
                    self.log("loop_error", err=f"{type(e).__name__}: {e}")

    def _install_handlers(self):
        def handler(signum, frame):
            self.log("interrupt", signal=int(signum))
            try:
                self.flush(final=True, force=True, run_final_hook=True,
                           msg=f"{self.run_id} interrupted (signal {signum})")
            finally:
                if signum == signal.SIGINT:
                    raise KeyboardInterrupt
                raise SystemExit(128 + int(signum))
        for sig in (signal.SIGINT, signal.SIGTERM):
            try:
                signal.signal(sig, handler)
            except Exception:
                pass
        atexit.register(self.close)

    def close(self):
        if self._closed:
            return
        self._closed = True; self.log("closing")
        self._stop.set(); self._wake.set()
        try:
            self._thread.join(timeout=5)
            self.flush(final=True, force=self._dirty.is_set(), run_final_hook=True)
        except Exception as e:
            print("Final Hugging Face sync failed:", type(e).__name__, e)
""",
 "crvs_data.py":    r"""
# crvs_data.py -- windowing, folds, normalisation and the torch Dataset.
# Shared by NB03, NB04 and NB05 so every experiment sees byte-identical inputs.
import json, math
import numpy as np
from pathlib import Path

CHANNELS = ["I", "Q", "phi", "dy", "vel", "acc", "amp", "cardiac"]
# One recording is stored as a single UNCOMPRESSED .npy of shape (len(ARRAY_ROWS), n).
# It has to be .npy, not .npz: np.load(..., mmap_mode="r") silently IGNORES mmap_mode on an
# .npz, so every __getitem__ would decompress all 11 arrays to slice 1024 samples out of
# each -- measured at 23 ms per window, which would dominate the GPU time on Kaggle.
ARRAY_ROWS = CHANNELS + ["ecg_norm", "peak_map", "rr_ms"]
ROW = {name: i for i, name in enumerate(ARRAY_ROWS)}
FS       = 128
# Bumped whenever this module changes in a way the notebooks depend on. Every notebook
# asserts it after import, because writing a .py and importing it is NOT idempotent inside
# one kernel: Python caches the module in sys.modules, so a second run silently keeps the
# first version. That is how a stale .npz loader survived a rebuilt notebook once already.
LIB_VERSION = 4
WINDOW   = 1024          # 8.0 s, frozen to Chowdhury et al. 2024 section 2.3.4
HOP_TRAIN = 512          # 50 % overlap on train only
SCENARIOS = ["Resting", "Valsalva", "Apnea", "Tilt-up", "Tilt-down"]

def canon_scenario(s):
    s = str(s).strip().lower()
    for key, out in [("tiltdown", "Tilt-down"), ("tilt_down", "Tilt-down"), ("tilt-down", "Tilt-down"),
                     ("tiltup", "Tilt-up"), ("tilt_up", "Tilt-up"), ("tilt-up", "Tilt-up"),
                     ("valsalva", "Valsalva"), ("apnea", "Apnea"), ("apnoea", "Apnea"),
                     ("rest", "Resting")]:
        if key in s:
            return out
    return str(s)

def range_normalise(x, eps=1e-8):
    # z-score then squash to [-1, 1]; the baseline used [0, 1], we declare the change
    x = np.asarray(x, np.float32)
    sd = float(x.std())
    if not np.isfinite(sd) or sd < eps:
        return np.zeros_like(x, np.float32)          # constant input -> 0, not -1
    x = (x - x.mean()) / (sd + eps)
    lo, hi = np.percentile(x, 0.5), np.percentile(x, 99.5)
    x = np.clip(x, lo, hi)
    rng = float(hi - lo)
    if rng < eps:
        return np.zeros_like(x, np.float32)
    return (2.0 * (x - lo) / rng - 1.0).astype(np.float32)

def peak_heatmap(n, peaks, sigma=3.0):
    # Gaussian bumps at each R peak -- the target for the multi-task peak head
    y = np.zeros(n, np.float32)
    if len(peaks) == 0:
        return y
    half = int(math.ceil(3 * sigma))
    g = np.exp(-0.5 * (np.arange(-half, half + 1) / sigma) ** 2).astype(np.float32)
    for p in np.asarray(peaks, int):
        a, b = max(0, p - half), min(n, p + half + 1)
        y[a:b] = np.maximum(y[a:b], g[a - (p - half): (b - (p - half))])
    return y

def rr_curve(n, peaks, fs=FS, lo_ms=300.0, hi_ms=2000.0):
    # per-sample instantaneous RR interval in ms, linearly interpolated between beats
    out = np.full(n, np.nan, np.float32)
    p = np.asarray(peaks, int)
    if len(p) < 3:
        return np.nan_to_num(out, nan=800.0)
    rr = np.diff(p) / fs * 1000.0
    mid = (p[:-1] + p[1:]) / 2.0
    ok = (rr > lo_ms) & (rr < hi_ms)
    if ok.sum() < 2:
        return np.nan_to_num(out, nan=float(np.median(rr)))
    out = np.interp(np.arange(n), mid[ok], rr[ok]).astype(np.float32)
    return out

_SLOW_WARNED = {"npz": False}

class _Rec:
    # Reads one recording in whichever format is on disk.
    #   .npy (preferred) -- uncompressed, genuinely memory-mapped, ~0.3 ms per window
    #   .npz (legacy)    -- what an earlier NB02 wrote; correct but ~85x slower, because
    #                       np.load ignores mmap_mode on a zip archive and every window
    #                       decompresses all 11 arrays.
    # Both are supported so an existing corpus keeps working without a 400 MB re-upload.
    __slots__ = ("data", "kind")

    def __init__(self, rec_dir, rid):
        d = Path(rec_dir)
        pnpy, pnpz = d / (rid + ".npy"), d / (rid + ".npz")
        if pnpy.exists():
            self.data = np.load(pnpy, mmap_mode="r"); self.kind = "npy"
        elif pnpz.exists():
            self.data = np.load(pnpz); self.kind = "npz"
            if not _SLOW_WARNED["npz"]:
                _SLOW_WARNED["npz"] = True
                print("  note: reading legacy .npz recordings. Correct, but about 85x slower "
                      "per window than .npy -- re-run NB02 to regenerate the corpus and cut "
                      "the data-loading cost.")
        else:
            raise FileNotFoundError(
                f"no recording for '{rid}' in {d} (looked for .npy and .npz). "
                "Either NB02 did not finish, or the snapshot_download allow_patterns in "
                "this notebook do not cover the format NB02 wrote.")

    def rows(self, names, s, e):
        if self.kind == "npy":
            return np.array(self.data[[ROW[n] for n in names], s:e], np.float32)
        return np.stack([np.array(self.data[n][s:e], np.float32) for n in names], 0)

    def one(self, name, s, e):
        if self.kind == "npy":
            return np.array(self.data[ROW[name], s:e], np.float32)
        return np.array(self.data[name][s:e], np.float32)

class WindowDataset:
    # Slices windows on the fly, so changing WINDOW or the overlap never requires
    # re-running NB02.
    def __init__(self, rec_dir, index, norm=None, channels=None, augment=False, seed=0):
        self.rec_dir = Path(rec_dir)
        self.index = index.reset_index(drop=True)
        self.norm = norm
        self.channels = channels or CHANNELS
        self.rows = [ROW[c] for c in self.channels]
        self.augment = augment
        self.seed = int(seed); self.epoch = 0
        self._cache = {}

    def set_epoch(self, epoch):
        # Augmentation is a pure function of (seed, epoch, index). With workers restarted
        # each epoch, an interrupted epoch can replay and skip batches byte-for-byte.
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.index)

    def _rec(self, rid):
        if rid not in self._cache:
            if len(self._cache) > 48:
                self._cache.pop(next(iter(self._cache)))
            self._cache[rid] = _Rec(self.rec_dir, rid)
        return self._cache[rid]

    def __getitem__(self, i):
        import torch
        r = self.index.iloc[i]
        z = self._rec(r["rec_id"])
        s, e = int(r["start"]), int(r["start"]) + WINDOW
        # _Rec.rows / _Rec.one always np.array (copy), never a view into a read-only
        # memmap -- torch.from_numpy on a non-writable array is undefined behaviour.
        x = z.rows(self.channels, s, e)
        if self.norm is not None:
            mu = np.asarray(self.norm["mean"], np.float32)[:, None]
            sd = np.asarray(self.norm["std"], np.float32)[:, None]
            x = (x - mu) / (sd + 1e-6)
        x = np.clip(x, -8.0, 8.0)
        y  = z.one("ecg_norm", s, e)
        pk = z.one("peak_map", s, e)
        rr = z.one("rr_ms", s, e) / 1000.0                           # seconds, O(1) scale
        if self.augment:
            rng = np.random.RandomState(np.random.SeedSequence(
                [self.seed, self.epoch, int(i)]).generate_state(1)[0])
            if rng.rand() < 0.5:
                x = x + rng.randn(*x.shape).astype(np.float32) * 0.01
            if rng.rand() < 0.3:
                g = np.float32(1.0 + 0.1 * rng.randn())
                x = x * g
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(y)[None, :],
                torch.from_numpy(pk)[None, :],
                torch.from_numpy(rr)[None, :])

def compute_norm(rec_dir, index, channels=CHANNELS, max_windows=4000, seed=0):
    # Per-channel mean/std computed on TRAIN WINDOWS ONLY. Computing them over the whole
    # corpus is a classic, invisible source of leakage.
    rng = np.random.RandomState(seed)
    idx = index if len(index) <= max_windows else index.iloc[
        rng.choice(len(index), max_windows, replace=False)]
    n = 0
    s1 = np.zeros(len(channels), np.float64)
    s2 = np.zeros(len(channels), np.float64)
    cache = {}
    rec_dir = Path(rec_dir)
    for _, r in idx.iterrows():
        rid = r["rec_id"]
        if rid not in cache:
            if len(cache) > 48:
                cache.pop(next(iter(cache)))
            cache[rid] = _Rec(rec_dir, rid)
        a, b = int(r["start"]), int(r["start"]) + WINDOW
        x = cache[rid].rows(list(channels), a, b).astype(np.float64)
        s1 += x.sum(1); s2 += (x * x).sum(1); n += x.shape[1]
    mean = s1 / max(n, 1)
    var = np.maximum(s2 / max(n, 1) - mean ** 2, 1e-12)
    return {"mean": mean.tolist(), "std": np.sqrt(var).tolist(),
            "n_samples": int(n), "channels": list(channels)}
""",
 "crvs_metrics.py": r"""
# crvs_metrics.py -- every metric the baseline reports, plus the ones it should have.
import numpy as np
from scipy import signal as ss
from scipy import stats as sstats

def _f(x):
    return np.nan_to_num(np.asarray(x, np.float64), nan=0.0, posinf=0.0, neginf=0.0)

def pearson(a, b):
    a, b = _f(a), _f(b)
    if a.std() < 1e-12 or b.std() < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])

def psd(x, fs=128, nperseg=256):
    f, p = ss.welch(_f(x), fs=fs, nperseg=min(nperseg, len(x)))
    return f, p

def seg_metrics(y, yhat, fs=128):
    # One window. Correlations are reported x100 to match the baseline's tables.
    y, yhat = _f(y), _f(yhat)
    mae = float(np.mean(np.abs(y - yhat)))
    mse = float(np.mean((y - yhat) ** 2))
    cct = 100.0 * pearson(y, yhat)
    _, py = psd(y, fs); _, ph = psd(yhat, fs)
    ccs = 100.0 * pearson(py, ph)
    rms = lambda v: float(np.sqrt(np.mean(np.asarray(v, np.float64) ** 2)))
    rr_t = rms(yhat - y) / (rms(y) + 1e-12)
    rr_s = rms(ph - py) / (rms(py) + 1e-12)
    return {"MAE": mae, "MSE": mse, "CC_temporal": cct, "CC_spectral": ccs,
            "RRMSE_temporal": rr_t, "RRMSE_spectral": rr_s,
            "R2": float(1.0 - np.sum((y - yhat) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-12))}

def detect_r_peaks(x, fs=128, refractory_s=0.25):
    x = _f(x)
    if len(x) < int(2 * fs):
        return np.array([], int)
    ny = fs / 2.0
    sos = ss.butter(4, [5.0 / ny, min(25.0, ny * 0.95) / ny], btype="band", output="sos")
    b = ss.sosfiltfilt(sos, x)
    e = np.convolve(np.diff(b, prepend=b[0]) ** 2,
                    np.ones(max(1, int(0.10 * fs))) / max(1, int(0.10 * fs)), "same")
    thr = np.percentile(e, 98) * 0.35
    pk, _ = ss.find_peaks(e, height=thr, distance=max(1, int(refractory_s * fs)))
    return pk

def hrv_from_peaks(pk, fs=128):
    out = {"n_peaks": int(len(pk)), "mean_rr_ms": np.nan, "sd_rr_ms": np.nan,
           "mean_hr_bpm": np.nan, "sd_hr_bpm": np.nan, "rmssd_ms": np.nan}
    if len(pk) < 4:
        return out
    rr = np.diff(np.asarray(pk, float)) / fs * 1000.0
    rr = rr[(rr > 300) & (rr < 2000)]
    if len(rr) < 3:
        return out
    hr = 60000.0 / rr
    out.update(mean_rr_ms=float(rr.mean()), sd_rr_ms=float(rr.std()),
               mean_hr_bpm=float(hr.mean()), sd_hr_bpm=float(hr.std()),
               rmssd_ms=float(np.sqrt(np.mean(np.diff(rr) ** 2))))
    return out

def peak_detection_scores(y, yhat, fs=128, tol_ms=100.0):
    # Match predicted R peaks to ground-truth peaks within a tolerance window.
    gt = detect_r_peaks(y, fs); pr = detect_r_peaks(yhat, fs)
    tol = tol_ms / 1000.0 * fs
    used = np.zeros(len(pr), bool)
    tp = 0
    errs = []
    for g in gt:
        if len(pr) == 0:
            break
        # float, not the int64 that find_peaks returns -- assigning np.inf into an
        # integer array raises OverflowError even when the mask selects nothing.
        d = np.abs(pr - g).astype(np.float64)
        d[used] = np.inf
        j = int(np.argmin(d))
        if d[j] <= tol:
            tp += 1; used[j] = True; errs.append((pr[j] - g) / fs * 1000.0)
    fp = int((~used).sum()); fn = int(len(gt) - tp)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return {"TP": tp, "FP": fp, "FN": fn, "precision": prec, "recall": rec, "F1": f1,
            "accuracy": tp / max(tp + fp + fn, 1),
            "timing_err_ms_median": float(np.median(np.abs(errs))) if errs else np.nan,
            "timing_err_ms_iqr": float(np.subtract(*np.percentile(np.abs(errs), [75, 25])))
                                  if len(errs) > 3 else np.nan,
            "missed_rate": fn / max(len(gt), 1)}

def aggregate(rows):
    import pandas as pd
    df = pd.DataFrame(rows)
    out = {}
    for c in df.columns:
        if df[c].dtype.kind in "fi":
            out[c] = float(df[c].mean()); out[c + "_std"] = float(df[c].std())
    return out

def bland_altman(a, b):
    a, b = _f(a), _f(b)
    m = (a + b) / 2.0; d = a - b
    bias = float(d.mean()); sd = float(d.std())
    return {"mean": m, "diff": d, "bias": bias, "sd": sd,
            "loa_lo": bias - 1.96 * sd, "loa_hi": bias + 1.96 * sd}

def wilcoxon_holm(groups, better="higher"):
    # Pairwise Wilcoxon signed-rank across folds, Holm-corrected. groups: {name: [values]}
    import itertools
    names = list(groups)
    raw = []
    for a, b in itertools.combinations(names, 2):
        x, y = np.asarray(groups[a], float), np.asarray(groups[b], float)
        n = min(len(x), len(y))
        if n < 3 or np.allclose(x[:n], y[:n]):
            raw.append((a, b, np.nan)); continue
        try:
            p = float(sstats.wilcoxon(x[:n], y[:n]).pvalue)
        except Exception:
            p = np.nan
        raw.append((a, b, p))
    ps = [r[2] for r in raw]
    order = np.argsort([p if np.isfinite(p) else 1.0 for p in ps])
    m = len(ps); adj = [np.nan] * m; run = 0.0
    for k, i in enumerate(order):
        p = ps[i]
        if not np.isfinite(p):
            continue
        run = max(run, (m - k) * p)
        adj[i] = min(1.0, run)
    return [{"a": raw[i][0], "b": raw[i][1], "p": ps[i], "p_holm": adj[i]} for i in range(m)]
""",
 "crvs_models.py":  r"""
# crvs_models.py -- the four baseline 1-D segmentation networks.
# All four are standardised the way Chowdhury et al. 2024 describe (section 3.1):
# 5 levels, 64 filters in the first level, doubling thereafter. Input (B, C_in, 1024).
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def cbr(i, o, k=3, s=1):
    return nn.Sequential(nn.Conv1d(i, o, k, s, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))

class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(cbr(i, o), cbr(o, o))
    def forward(self, x):
        return self.b(x)

# ------------------------------------------------------------------ UNet
class UNet1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.inc = DoubleConv(in_ch, chs[0])
        self.downs = nn.ModuleList()
        for i in range(levels - 1):
            self.downs.append(DoubleConv(chs[i], chs[i + 1]))
        self.bott = DoubleConv(chs[-1], chs[-1] * 2)
        self.ups = nn.ModuleList()
        self.decs = nn.ModuleList()
        prev = chs[-1] * 2
        for c in reversed(chs):
            self.ups.append(nn.ConvTranspose1d(prev, c, 4, 2, 1))
            self.decs.append(DoubleConv(c * 2, c))
            prev = c
        self.head = nn.Conv1d(chs[0], out_ch, 1)
    def forward(self, x):
        skips = []
        h = self.inc(x); skips.append(h)
        for d in self.downs:
            h = d(F.max_pool1d(h, 2)); skips.append(h)
        h = self.bott(F.max_pool1d(h, 2))
        for up, dec, sk in zip(self.ups, self.decs, reversed(skips)):
            h = up(h)
            if h.shape[-1] != sk.shape[-1]:
                h = F.interpolate(h, size=sk.shape[-1], mode="linear", align_corners=False)
            h = dec(torch.cat([h, sk], 1))
        return {"wave": torch.tanh(self.head(h))}

# ------------------------------------------------------------------ LinkNet
class LinkEnc(nn.Module):
    def __init__(self, i, o, stride=2):
        super().__init__()
        self.c1 = cbr(i, o, 3, stride)
        self.c2 = nn.Sequential(nn.Conv1d(o, o, 3, 1, 1, bias=False), nn.BatchNorm1d(o))
        self.sc = nn.Sequential(nn.Conv1d(i, o, 1, stride, bias=False), nn.BatchNorm1d(o))
    def forward(self, x):
        return F.relu(self.c2(self.c1(x)) + self.sc(x))

class LinkDec(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        m = max(i // 4, 8)
        self.a = cbr(i, m, 1)
        self.b = nn.Sequential(nn.ConvTranspose1d(m, m, 4, 2, 1, bias=False),
                               nn.BatchNorm1d(m), nn.ReLU(inplace=True))
        self.c = cbr(m, o, 1)
    def forward(self, x):
        return self.c(self.b(self.a(x)))

class LinkNet1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, chs[0], 7, 1)
        self.encs = nn.ModuleList()
        prev = chs[0]
        for c in chs:
            self.encs.append(LinkEnc(prev, c, 2)); prev = c
        self.bott = cbr(prev, prev)
        self.decs = nn.ModuleList()
        rev = list(reversed(chs))
        for k, c in enumerate(rev):
            nxt = rev[k + 1] if k + 1 < len(rev) else chs[0]
            self.decs.append(LinkDec(c, nxt))
        self.head = nn.Sequential(cbr(chs[0], chs[0]), nn.Conv1d(chs[0], out_ch, 1))
    def forward(self, x):
        h = self.stem(x)
        skips = []
        for e in self.encs:
            h = e(h); skips.append(h)
        h = self.bott(h)
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 2 - k
            if j >= 0:
                s = skips[j]
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                h = h + s
        return {"wave": torch.tanh(self.head(h))}

# ------------------------------------------------------------------ FPN
class FPN1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4, pyr=128):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, chs[0], 7, 1)
        self.encs = nn.ModuleList()
        prev = chs[0]
        for c in chs:
            self.encs.append(LinkEnc(prev, c, 2)); prev = c
        self.lat = nn.ModuleList([nn.Conv1d(c, pyr, 1) for c in chs])
        self.smooth = nn.ModuleList([cbr(pyr, pyr) for _ in chs])
        self.heads = nn.ModuleList([nn.Sequential(cbr(pyr, pyr // 2), cbr(pyr // 2, pyr // 2))
                                    for _ in chs])
        self.head = nn.Sequential(cbr(pyr // 2, pyr // 2), nn.Conv1d(pyr // 2, out_ch, 1))
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(x); feats = []
        for e in self.encs:
            h = e(h); feats.append(h)
        ps = [None] * len(feats)
        ps[-1] = self.lat[-1](feats[-1])
        for i in range(len(feats) - 2, -1, -1):
            up = F.interpolate(ps[i + 1], size=feats[i].shape[-1], mode="linear",
                               align_corners=False)
            ps[i] = self.lat[i](feats[i]) + up
        ps = [s(p) for s, p in zip(self.smooth, ps)]
        acc = None
        for hd, p in zip(self.heads, ps):
            v = F.interpolate(hd(p), size=L, mode="linear", align_corners=False)
            acc = v if acc is None else acc + v
        return {"wave": torch.tanh(self.head(acc))}

# ------------------------------------------------------------------ MultiResLinkNet
class MultiResBlock(nn.Module):
    # MultiResUNet block (Ibtehaz & Rahman) in 1-D: three successive 3-conv stages of
    # increasing width, concatenated, plus a 1x1 residual shortcut.
    def __init__(self, cin, U, alpha=1.67):
        super().__init__()
        W = alpha * U
        # max(1, ...): below U=4 the 0.167 stage floors to zero channels, and the failure
        # then surfaces as an opaque conv error rather than pointing here.
        c1, c2, c3 = (max(1, int(W * 0.167)), max(1, int(W * 0.333)), max(1, int(W * 0.5)))
        self.out_channels = c1 + c2 + c3
        self.sc = nn.Sequential(nn.Conv1d(cin, self.out_channels, 1, bias=False),
                                nn.BatchNorm1d(self.out_channels))
        self.a = cbr(cin, c1); self.b = cbr(c1, c2); self.c = cbr(c2, c3)
        self.bn1 = nn.BatchNorm1d(self.out_channels)
        self.bn2 = nn.BatchNorm1d(self.out_channels)
    def forward(self, x):
        s = self.sc(x)
        a = self.a(x); b = self.b(a); c = self.c(b)
        o = self.bn1(torch.cat([a, b, c], 1))
        return F.relu(self.bn2(o + s))

class ResPath(nn.Module):
    # Processes an encoder feature before it is added to the decoder, instead of a raw skip.
    def __init__(self, ch, length):
        super().__init__()
        self.blocks = nn.ModuleList()
        for _ in range(max(1, length)):
            self.blocks.append(nn.ModuleDict({
                "sc": nn.Sequential(nn.Conv1d(ch, ch, 1, bias=False), nn.BatchNorm1d(ch)),
                "cv": nn.Sequential(nn.Conv1d(ch, ch, 3, padding=1, bias=False),
                                    nn.BatchNorm1d(ch)),
            }))
    def forward(self, x):
        for b in self.blocks:
            x = F.relu(b["sc"](x) + b["cv"](x))
        return x

class MultiResLinkNet1D(nn.Module):
    # LinkNet skeleton, MultiRes blocks instead of plain convolutions, ResPath skips added
    # (not concatenated), and deep supervision from every encoder level.
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4, deep_supervision=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        units = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, base, 7, 1)
        self.encs = nn.ModuleList(); self.paths = nn.ModuleList()
        prev = base; enc_ch = []
        for i, u in enumerate(units):
            blk = MultiResBlock(prev, u)
            self.encs.append(blk)
            self.paths.append(ResPath(blk.out_channels, levels - i))
            enc_ch.append(blk.out_channels); prev = blk.out_channels
        self.bott = MultiResBlock(prev, units[-1])
        self.decs = nn.ModuleList()
        rev_ch = list(reversed(enc_ch))
        cur = self.bott.out_channels
        # Decoder step k must emerge with the channel count AND length of skips[-1-k], or
        # the additive skip is silently dropped and every ResPath receives zero gradient.
        # Encoder here pools AFTER appending the skip, so the target is rev_ch[k] -- not
        # rev_ch[k+1], which is correct only for the stride-2 encoder in LinkNet1D.
        for k in range(levels):
            tgt = rev_ch[k]
            self.decs.append(LinkDec(cur, tgt)); cur = tgt
        self.head = nn.Sequential(cbr(cur, base), nn.Conv1d(base, out_ch, 1))
        self.aux = nn.ModuleList([nn.Conv1d(c, out_ch, 1) for c in enc_ch]) \
                   if deep_supervision else None
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(x)
        skips = []
        for e in self.encs:
            h = e(h); skips.append(h)
            h = F.max_pool1d(h, 2)
        h = self.bott(h)
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 1 - k
            if j >= 0:
                s = self.paths[j](skips[j])
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                if h.shape[1] != s.shape[1]:
                    raise RuntimeError(                     # raise, not assert: an invariant
                        f"skip channel mismatch at decoder {k}: {h.shape[1]} vs "        # this
                        f"{s.shape[1]}. Dropping it silently is what cost 59% of this "  # load
                        "model's gradient once already.")   # bearing must survive python -O
                h = h + s
        if h.shape[-1] != L:
            h = F.interpolate(h, size=L, mode="linear", align_corners=False)
        out = {"wave": torch.tanh(self.head(h))}
        if self.aux is not None and self.training:
            out["aux"] = [F.interpolate(a(s), size=L, mode="linear", align_corners=False)
                          for a, s in zip(self.aux, skips)]
        return out

BASELINES = {"fpn": FPN1D, "unet": UNet1D, "linknet": LinkNet1D,
             "multireslinknet": MultiResLinkNet1D}

def build_baseline(name, in_ch=1, out_ch=1, base=64, levels=4):
    return BASELINES[name](in_ch=in_ch, out_ch=out_ch, base=base, levels=levels)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)
""",
 "crvs_cmnet.py":   r"""
# crvs_cmnet.py -- CardioMamba-Net (contributions C1-C5 of PLAN.md).
# C2 dual-domain encoder, C3 bidirectional SSM bottleneck, C4 multi-task decoder with
# peak-conditioned FiLM refinement. C1 lives in the data pipeline, C5 in crvs_losses.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from crvs_models import cbr, MultiResBlock, ResPath, LinkDec, count_params

class ECA(nn.Module):
    # Efficient channel attention: a length-k 1-D conv over the channel descriptor.
    def __init__(self, ch, k=5):
        super().__init__()
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)
    def forward(self, x):
        w = x.mean(-1, keepdim=True).transpose(1, 2)
        w = torch.sigmoid(self.conv(w)).transpose(1, 2)
        return x * w

class LiftingUnit(nn.Module):
    # Learnable second-generation wavelet: split into even/odd, predict, update.
    # Replaces a fixed wavelet basis with one the network chooses for radar.
    def __init__(self, ch, k=5):
        super().__init__()
        self.P = nn.Sequential(nn.Conv1d(ch, ch, k, padding=k // 2), nn.Tanh(),
                               nn.Conv1d(ch, ch, 1))
        self.U = nn.Sequential(nn.Conv1d(ch, ch, k, padding=k // 2), nn.Tanh(),
                               nn.Conv1d(ch, ch, 1))
    def forward(self, x):
        xe, xo = x[..., ::2], x[..., 1::2]
        n = min(xe.shape[-1], xo.shape[-1])
        xe, xo = xe[..., :n], xo[..., :n]
        d = xo - self.P(xe)
        c = xe + self.U(d)
        return c, d

class WaveletBranch(nn.Module):
    # Multi-resolution analysis producing one feature map per scale, to sit alongside the
    # convolutional branch. LifWavNet uses this idea as the whole network; here it is half
    # of a dual-domain encoder.
    #
    # Level 0 is taken at the INPUT resolution, before any lifting. Encoder level i sits at
    # L/2^i, and a lifting unit halves length, so starting the branch with a lifting step
    # would put every wavelet feature one octave below its conv counterpart and force a 2x
    # upsample at every fusion. The dual-domain claim (C2) is that the two branches see the
    # SAME scale from different domains, so they have to be aligned octave for octave.
    def __init__(self, ch, out_chs, levels=4):
        super().__init__()
        self.proj0 = nn.Conv1d(ch, out_chs[0], 1)
        self.units = nn.ModuleList([LiftingUnit(ch) for _ in range(max(levels - 1, 0))])
        self.proj = nn.ModuleList([nn.Conv1d(ch * 2, o, 1) for o in out_chs[1:]])
    def forward(self, x):
        feats = [self.proj0(x)]
        c = x
        for u, p in zip(self.units, self.proj):
            c, d = u(c)
            feats.append(p(torch.cat([c, d], 1)))
        return feats

class S4D(nn.Module):
    # Diagonal state-space layer (S4D-Lin). Pure PyTorch: an FFT convolution with a kernel
    # built from learned diagonal dynamics. No custom CUDA, so it always builds on Kaggle.
    def __init__(self, d_model, d_state=64, dt_min=1e-3, dt_max=1e-1):
        super().__init__()
        H, N = d_model, d_state // 2
        log_dt = torch.rand(H) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        self.log_dt = nn.Parameter(log_dt)
        self.log_A_real = nn.Parameter(torch.log(0.5 * torch.ones(H, N)))
        self.A_imag = nn.Parameter(math.pi * torch.arange(N).float().repeat(H, 1))
        self.C = nn.Parameter(torch.randn(H, N, 2) * (0.5 ** 0.5))
        self.D = nn.Parameter(torch.randn(H))
    def kernel(self, L, device, dtype=torch.float32):
        dt = torch.exp(self.log_dt).to(dtype).unsqueeze(-1)
        A = -torch.exp(self.log_A_real.to(dtype)) + 1j * self.A_imag.to(dtype)
        C = torch.view_as_complex(self.C.to(dtype).contiguous())
        dtA = A * dt
        n = torch.arange(L, device=device, dtype=dtype)
        K = dtA.unsqueeze(-1) * n
        Cc = C * (torch.exp(dtA) - 1.0) / A
        return 2.0 * torch.einsum("hn,hnl->hl", Cc, torch.exp(K)).real
    def forward(self, u):
        L = u.shape[-1]
        uf = u.float()
        k = self.kernel(L, u.device)
        n = 2 * L
        y = torch.fft.irfft(torch.fft.rfft(uf, n=n) * torch.fft.rfft(k, n=n), n=n)[..., :L]
        y = y + uf * self.D.unsqueeze(-1)
        return y.to(u.dtype)

class BiSSM(nn.Module):
    # Bidirectional SSM block: an 8 s window holds 8-10 cardiac cycles, and this is what
    # lets beat n inform beat n+1. Linear time in sequence length.
    def __init__(self, d, d_state=64, expand=2, dropout=0.1):
        super().__init__()
        self.n1 = nn.LayerNorm(d)
        self.fwd = S4D(d, d_state)
        self.bwd = S4D(d, d_state)
        self.mix = nn.Conv1d(2 * d, d, 1)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Conv1d(d, expand * d, 1), nn.GELU(),
                                nn.Dropout(dropout), nn.Conv1d(expand * d, d, 1))
    def forward(self, x):
        h = self.n1(x.transpose(1, 2)).transpose(1, 2)
        f = self.fwd(h)
        b = self.bwd(h.flip(-1)).flip(-1)
        x = x + self.mix(torch.cat([f, b], 1))
        h = self.n2(x.transpose(1, 2)).transpose(1, 2)
        return x + self.ff(h)

class TransformerBottleneck(nn.Module):
    # The fair-fight control for C3: same budget, attention instead of an SSM.
    def __init__(self, d, nhead=8, layers=3, dropout=0.1, max_len=1024):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, max_len, d) * 0.02)
        lyr = nn.TransformerEncoderLayer(d, nhead, dim_feedforward=2 * d, dropout=dropout,
                                         batch_first=True, norm_first=True,
                                         activation="gelu")
        self.enc = nn.TransformerEncoder(lyr, layers)
    def forward(self, x):
        h = x.transpose(1, 2)
        h = h + self.pos[:, :h.shape[1]]
        return self.enc(h).transpose(1, 2)

class FiLM(nn.Module):
    # Peak-conditioned refinement: the R-peak head tells the waveform head where a QRS
    # belongs BEFORE it draws one. Modulation is per-sample, because a QRS is localised.
    def __init__(self, cond_ch, feat_ch, k=9):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(cond_ch, feat_ch, k, padding=k // 2), nn.GELU(),
                                 nn.Conv1d(feat_ch, 2 * feat_ch, 1))
    def forward(self, feat, cond):
        g, b = self.net(cond).chunk(2, 1)
        return feat * (1.0 + torch.tanh(g)) + b

class CardioMambaNet(nn.Module):
    def __init__(self, in_ch=8, base=32, levels=4, d_ssm=256, ssm_blocks=3, d_state=64,
                 bottleneck="ssm", use_wavelet=True, multitask=True, use_film=True,
                 dropout=0.1):
        super().__init__()
        self.use_wavelet = use_wavelet
        self.multitask = multitask
        self.use_film = use_film and multitask
        units = [base * (2 ** i) for i in range(levels)]
        # C1 lands here: a learnable 1x1 mix over the 8 physics channels, so the network
        # can rediscover arctangent demodulation if that really is optimal.
        self.mix = nn.Sequential(nn.Conv1d(in_ch, 32, 1), nn.GELU())
        self.stem = cbr(32, base, 7, 1)
        self.encs = nn.ModuleList(); self.paths = nn.ModuleList()
        prev = base; enc_ch = []
        for i, u in enumerate(units):
            blk = MultiResBlock(prev, u)
            self.encs.append(blk)
            self.paths.append(ResPath(blk.out_channels, levels - i))
            enc_ch.append(blk.out_channels); prev = blk.out_channels
        if use_wavelet:
            self.wave = WaveletBranch(base, enc_ch, levels)
            self.fuse = nn.ModuleList([nn.Sequential(nn.Conv1d(2 * c, c, 1), ECA(c))
                                       for c in enc_ch])
        self.pre = nn.Conv1d(prev, d_ssm, 1)
        if bottleneck == "ssm":
            self.bott = nn.Sequential(*[BiSSM(d_ssm, d_state, dropout=dropout)
                                        for _ in range(ssm_blocks)])
        elif bottleneck == "transformer":
            self.bott = TransformerBottleneck(d_ssm, layers=ssm_blocks, dropout=dropout)
        else:
            self.bott = nn.Sequential(cbr(d_ssm, d_ssm), cbr(d_ssm, d_ssm))
        self.post = nn.Conv1d(d_ssm, prev, 1)
        self.decs = nn.ModuleList()
        rev = list(reversed(enc_ch)); cur = prev
        # Same indexing rule as MultiResLinkNet1D: target rev[k], so decoder step k lines up
        # with skips[-1-k] in both channels and length and the ResPath actually contributes.
        for k in range(levels):
            tgt = rev[k]
            self.decs.append(LinkDec(cur, tgt)); cur = tgt
        self.refine = cbr(cur, base)
        self.head_wave = nn.Conv1d(base, 1, 1)
        if multitask:
            self.head_peak = nn.Sequential(cbr(cur, base), nn.Conv1d(base, 1, 1))
            rr_ch = max(base // 2, 4)
            self.head_rr = nn.Sequential(cbr(cur, rr_ch), nn.Conv1d(rr_ch, 1, 1))
            if self.use_film:
                self.film = FiLM(1, base)
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(self.mix(x))
        wfeat = self.wave(h) if self.use_wavelet else None
        skips = []
        for i, e in enumerate(self.encs):
            h = e(h)
            if wfeat is not None:
                w = wfeat[i]
                if w.shape[-1] != h.shape[-1]:
                    w = F.interpolate(w, size=h.shape[-1], mode="linear", align_corners=False)
                h = self.fuse[i](torch.cat([h, w], 1))
            skips.append(h)
            h = F.max_pool1d(h, 2)
        h = self.post(self.bott(self.pre(h)))
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 1 - k
            if j >= 0:
                s = self.paths[j](skips[j])
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                if h.shape[1] != s.shape[1]:
                    raise RuntimeError(
                        f"skip channel mismatch at decoder {k}: {h.shape[1]} vs {s.shape[1]}")
                h = h + s
        if h.shape[-1] != L:
            h = F.interpolate(h, size=L, mode="linear", align_corners=False)
        out = {}
        if self.multitask:
            peak_logit = self.head_peak(h)
            out["peak"] = peak_logit
            out["rr"] = F.softplus(self.head_rr(h))
            f = self.refine(h)
            if self.use_film:
                # Gradient flows through the conditioning on purpose: the peak head is meant
                # to be shaped by the waveform loss as well as its own, which is the point of
                # peak-conditioned refinement. Detaching here would make it a one-way hint.
                f = self.film(f, torch.sigmoid(peak_logit))
            out["wave"] = torch.tanh(self.head_wave(f))
        else:
            out["wave"] = torch.tanh(self.head_wave(self.refine(h)))
        return out

def build_cmnet(**kw):
    return CardioMambaNet(**kw)
""",
 "crvs_losses.py":  r"""
# crvs_losses.py -- C5, the morphology-aware composite loss.
# Plain MSE is the conditional mean, so it flattens the R peak; that is exactly why the
# baseline over-estimates RMSSD by ~2x. Every term here exists to stop that.
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiResSTFTLoss(nn.Module):
    # Spectral convergence + log-magnitude at three resolutions. Forces the model to get
    # the spectrum right, not just the sample-wise average.
    def __init__(self, ffts=(256, 128, 64)):
        super().__init__()
        self.ffts = ffts
    def _one(self, y, yh, n):
        hop, win = n // 4, n
        w = torch.hann_window(win, device=y.device, dtype=torch.float32)
        kw = dict(n_fft=n, hop_length=hop, win_length=win, window=w,
                  return_complex=True, center=True, pad_mode="reflect")
        Y = torch.stft(y, **kw).abs().clamp_min(1e-7)
        H = torch.stft(yh, **kw).abs().clamp_min(1e-7)
        sc = torch.norm(Y - H, p="fro", dim=(-2, -1)) / (torch.norm(Y, p="fro", dim=(-2, -1)) + 1e-7)
        mag = F.l1_loss(torch.log(H), torch.log(Y))
        return sc.mean() + mag
    def forward(self, y, yh):
        y = y.squeeze(1).float(); yh = yh.squeeze(1).float()
        return sum(self._one(y, yh, n) for n in self.ffts) / len(self.ffts)

def pearson_loss(y, yh, eps=1e-8):
    y = y.squeeze(1).float(); yh = yh.squeeze(1).float()
    y = y - y.mean(-1, keepdim=True); yh = yh - yh.mean(-1, keepdim=True)
    num = (y * yh).sum(-1)
    den = y.norm(dim=-1) * yh.norm(dim=-1) + eps
    return (1.0 - num / den).mean()

def focal_bce(logit, target, alpha=0.75, gamma=2.0):
    p = torch.sigmoid(logit)
    ce = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    pt = p * target + (1 - p) * (1 - target)
    w = alpha * target + (1 - alpha) * (1 - target)
    return (w * (1 - pt).pow(gamma) * ce).mean()

class CompositeLoss(nn.Module):
    def __init__(self, w_huber=1.0, w_stft=0.5, w_peak=0.3, w_rr=0.1,
                 w_peakw=0.5, w_corr=0.3, huber_delta=0.1, peak_weight=4.0):
        super().__init__()
        self.w = dict(huber=w_huber, stft=w_stft, peak=w_peak, rr=w_rr,
                      peakw=w_peakw, corr=w_corr)
        self.delta = huber_delta
        self.peak_weight = peak_weight
        self.stft = MultiResSTFTLoss()
    def forward(self, pred, y, pk=None, rr=None):
        parts = {}
        wave = pred["wave"]
        if self.w["huber"]:
            parts["huber"] = F.huber_loss(wave, y, delta=self.delta)
        if self.w["stft"]:
            parts["stft"] = self.stft(y, wave)
        if self.w["corr"]:
            parts["corr"] = pearson_loss(y, wave)
        if self.w["peakw"] and pk is not None:
            wgt = 1.0 + self.peak_weight * pk
            parts["peakw"] = ((wgt * (wave - y).abs()).sum() / (wgt.sum() + 1e-8))
        if self.w["peak"] and pk is not None and "peak" in pred:
            parts["peak"] = focal_bce(pred["peak"], pk)
        if self.w["rr"] and rr is not None and "rr" in pred:
            parts["rr"] = F.l1_loss(pred["rr"], rr)
        if "aux" in pred:
            parts["aux"] = sum(F.huber_loss(a, y, delta=self.delta)
                               for a in pred["aux"]) / max(len(pred["aux"]), 1) * 0.2
        total = sum(self.w.get(k, 1.0) * v for k, v in parts.items())
        return total, {k: float(v.detach()) for k, v in parts.items()}

class MSEOnly(nn.Module):
    # The baseline's objective, kept verbatim so ablation row 1 is a true reproduction.
    def forward(self, pred, y, pk=None, rr=None):
        l = F.mse_loss(pred["wave"], y)
        if "aux" in pred:
            l = l + 0.2 * sum(F.mse_loss(a, y) for a in pred["aux"]) / max(len(pred["aux"]), 1)
        return l, {"mse": float(l.detach())}
""",
 "crvs_engine.py":  r"""
# crvs_engine.py -- deterministic dual-GPU engine with mid-epoch recovery and telemetry.
import csv, json, math, time, os, random, shutil, subprocess, threading, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ENGINE_VERSION = 5

def _autocast(device_type, enabled):
    try:
        return torch.amp.autocast(device_type=device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)

def _grad_scaler(device_type, enabled):
    try:
        return torch.amp.GradScaler(device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

def pick_device():
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        names = [torch.cuda.get_device_name(i) for i in range(n)]
        return torch.device("cuda"), n, names
    return torch.device("cpu"), 0, []

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def _atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2, default=str, allow_nan=True)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def _jsonable(v):
    if isinstance(v, (np.floating, np.integer)): return v.item()
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, float) and not math.isfinite(v): return None
    return v

def _append_jsonl(path, record):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    clean = {str(k): _jsonable(v) for k, v in record.items()}
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(clean, default=str, allow_nan=False) + "\n"); f.flush()

def _mean_parts(sums, n, prefix):
    return {f"{prefix}_{k}": float(v) / max(int(n), 1) for k, v in sums.items()}

def _system_stats(out_dir):
    d = shutil.disk_usage(Path(out_dir))
    rec = {"disk_free_gb": d.free / 2**30, "disk_used_gb": d.used / 2**30}
    try:
        import resource
        rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        rec["process_peak_rss_gb"] = rss / 2**20  # Linux ru_maxrss is KiB
    except Exception:
        pass
    try:
        load = os.getloadavg(); rec.update(cpu_load_1m=load[0], cpu_load_5m=load[1], cpu_load_15m=load[2])
    except Exception:
        pass
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            rec[f"gpu{i}_peak_alloc_gb"] = torch.cuda.max_memory_allocated(i) / 2**30
            rec[f"gpu{i}_peak_reserved_gb"] = torch.cuda.max_memory_reserved(i) / 2**30
        try:
            q = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,temperature.gpu,power.draw,memory.used",
                                "--format=csv,noheader,nounits"], capture_output=True, text=True,
                               timeout=10, check=False)
            for line in q.stdout.strip().splitlines():
                vals = [x.strip() for x in line.split(",")]
                if len(vals) == 5:
                    i = vals[0]
                    for key, val in zip(("util_pct", "temp_c", "power_w", "mem_used_mb"), vals[1:]):
                        try: rec[f"gpu{i}_{key}"] = float(val)
                        except ValueError: pass
        except Exception:
            pass
    return rec

class Trainer:
    def __init__(self, model, loss_fn, out_dir, run_id, sync=None, lr=5e-4, weight_decay=1e-4,
                 epochs=120, patience=20, batch_size=64, num_workers=2, amp=True,
                 multi_gpu=True, grad_clip=1.0, min_lr=1e-6, log_every=25,
                 checkpoint_every_steps=50, checkpoint_every_s=300, seed=42,
                 require_dual_gpu=False, run_config=None, monitor="val_total",
                 monitor_mode="min", min_epochs=0):
        self.device, self.ngpu, self.gpu_names = pick_device()
        if require_dual_gpu and self.ngpu < 2:
            raise RuntimeError("This training notebook requires Kaggle GPU T4 x2. "
                               "Choose Settings > Accelerator > GPU T4 x2, then restart.")
        self.raw_model = model.to(self.device); self.model = self.raw_model
        if multi_gpu and self.ngpu > 1:
            self.model = nn.DataParallel(self.raw_model)
        self.loss_fn = loss_fn
        self.out = Path(out_dir); self.out.mkdir(parents=True, exist_ok=True)
        self.run_id = run_id; self.sync = sync
        self.epochs = int(epochs); self.patience = int(patience)
        self.monitor = str(monitor)
        self.monitor_mode = str(monitor_mode).lower()
        if self.monitor_mode not in ("min", "max"):
            raise ValueError("monitor_mode must be 'min' or 'max'")
        self.min_epochs = max(0, int(min_epochs))
        self.bs = int(batch_size); self.nw = int(num_workers); self.seed = int(seed)
        self.amp = bool(amp and self.device.type == "cuda"); self.grad_clip = grad_clip
        self.opt = torch.optim.AdamW(self.raw_model.parameters(), lr=lr, weight_decay=weight_decay)
        self.sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.opt, T_max=self.epochs, eta_min=min_lr)
        self.scaler = _grad_scaler(self.device.type, self.amp)
        self.log_every = max(1, int(log_every))
        self.checkpoint_every_steps = max(1, int(checkpoint_every_steps))
        self.checkpoint_every_s = max(30, int(checkpoint_every_s))
        self.config = dict(run_config or {})
        self.config_hash = hashlib.sha256(json.dumps(
            self.config, sort_keys=True, default=str).encode()).hexdigest()
        self.state = {"schema_version": 4, "engine_version": ENGINE_VERSION,
                      "epoch": 0, "active_epoch": 0, "batch_in_epoch": 0,
                      "global_step": 0,
                      "best": float("inf") if self.monitor_mode == "min" else -float("inf"),
                      "best_epoch": -1, "monitor": self.monitor,
                      "monitor_mode": self.monitor_mode, "min_epochs": self.min_epochs,
                      "bad_epochs": 0, "history": [], "partial": {},
                      "run_id": run_id, "done": False, "config_hash": self.config_hash,
                      "created_utc": datetime.now(timezone.utc).isoformat()}
        self._save_lock = threading.Lock(); self._last_checkpoint = time.time()
        self._active = False
        _atomic_json(self.out / "run_config.json", self.config)
        _atomic_json(self.out / "environment.json", {
            "engine_version": ENGINE_VERSION, "torch": torch.__version__,
            "cuda": torch.version.cuda, "gpu_count": self.ngpu, "gpu_names": self.gpu_names,
            "amp": self.amp, "python": os.sys.version, "config_hash": self.config_hash})

    @property
    def ckpt(self):
        return self.out / "state.pt"

    def _payload(self):
        return {"model": self.raw_model.state_dict(), "opt": self.opt.state_dict(),
                "sched": self.sched.state_dict(), "scaler": self.scaler.state_dict(),
                "state": self.state, "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "np_rng": np.random.get_state(), "python_rng": random.getstate(),
                "config": self.config, "config_hash": self.config_hash,
                "saved_utc": datetime.now(timezone.utc).isoformat()}

    def save(self, tag="state", reason="checkpoint"):
        with self._save_lock:
            path = self.out / (tag + ".pt"); tmp = path.with_suffix(".pt.tmp")
            torch.save(self._payload(), tmp); os.replace(tmp, path)
            _atomic_json(self.out / "state.json", self.state)
            self._last_checkpoint = time.time()
        if self.sync:
            self.sync.mark_dirty(f"{self.run_id}:{reason}")

    def emergency_checkpoint(self):
        if self._active:
            self.save("state", reason="interrupt-emergency")

    def load(self):
        if not self.ckpt.exists():
            return False
        try:
            d = torch.load(self.ckpt, map_location=self.device, weights_only=False)
            got_hash = d.get("config_hash", d.get("state", {}).get("config_hash"))
            if got_hash and got_hash != self.config_hash:
                raise RuntimeError("checkpoint configuration differs from this run. "
                                   "Use a new RUN_ID or restore the original configuration.")
            self.raw_model.load_state_dict(d["model"], strict=True)
            self.opt.load_state_dict(d["opt"]); self.sched.load_state_dict(d["sched"])
            self.scaler.load_state_dict(d["scaler"]); self.state = d["state"]
            torch.set_rng_state(d["torch_rng"].cpu()); np.random.set_state(d["np_rng"])
            random.setstate(d["python_rng"])
            if torch.cuda.is_available() and d.get("cuda_rng"):
                torch.cuda.set_rng_state_all([x.cpu() for x in d["cuda_rng"]])
            print(f"  resumed {self.run_id}: completed_epoch={self.state['epoch']}, "
                  f"active_epoch={self.state.get('active_epoch')}, "
                  f"completed_batches={self.state.get('batch_in_epoch', 0)}")
            return True
        except Exception as e:
            raise RuntimeError(f"Checkpoint exists but cannot be resumed safely: "
                               f"{type(e).__name__}: {e}") from e

    def _loader(self, ds, shuffle, epoch=0):
        if len(ds) == 0:
            raise RuntimeError("empty dataset -- check the subject split")
        if hasattr(ds, "set_epoch"):
            ds.set_epoch(epoch)
        drop = bool(shuffle) and len(ds) > self.bs
        gen = torch.Generator(); gen.manual_seed(self.seed + int(epoch) * 1000003)
        return DataLoader(ds, batch_size=min(self.bs, max(len(ds), 1)), shuffle=shuffle,
                          generator=gen, num_workers=self.nw,
                          pin_memory=(self.device.type == "cuda"), drop_last=drop,
                          persistent_workers=False)

    def _step(self, batch, train):
        x, y, pk, rr = [b.to(self.device, non_blocking=True) for b in batch]
        with _autocast(self.device.type, self.amp):
            pred = self.model(x)
            if isinstance(pred, dict) and "aux" in pred and not train:
                pred = {k: v for k, v in pred.items() if k != "aux"}
            loss, parts = self.loss_fn(pred, y, pk, rr)
        return loss, {k: float(v) for k, v in parts.items()}, pred, y, pk, rr

    def _write_batch(self, rec):
        _append_jsonl(self.out / "batch_metrics.jsonl", rec)

    def _validation(self, val_ds, epoch):
        self.model.eval(); sums = {}; n = 0; Y = []; P = []; PK = []; PH = []; RR = []; RH = []
        t0 = time.time()
        with torch.no_grad():
            for batch in self._loader(val_ds, False, epoch):
                loss, parts, pred, y, pk, rr = self._step(batch, False)
                parts = {"total": float(loss), **parts}
                for k, v in parts.items(): sums[k] = sums.get(k, 0.0) + float(v)
                n += 1; Y.append(y.squeeze(1).float().cpu().numpy())
                P.append(pred["wave"].squeeze(1).float().cpu().numpy())
                PK.append(pk.squeeze(1).float().cpu().numpy())
                if "peak" in pred: PH.append(pred["peak"].squeeze(1).float().cpu().numpy())
                RR.append(rr.squeeze(1).float().cpu().numpy())
                if "rr" in pred: RH.append(pred["rr"].squeeze(1).float().cpu().numpy())
        if n == 0: raise RuntimeError("validation loader yielded zero batches")
        y = np.concatenate(Y); p = np.concatenate(P); pk = np.concatenate(PK)
        rec = _mean_parts(sums, n, "val")
        rec["val_seconds"] = time.time() - t0; rec["val_windows"] = len(y)
        rec.update(val_true_mean=float(y.mean()), val_true_std=float(y.std()),
                   val_true_min=float(y.min()), val_true_max=float(y.max()),
                   val_pred_mean=float(p.mean()), val_pred_std=float(p.std()),
                   val_pred_min=float(p.min()), val_pred_max=float(p.max()),
                   val_pred_bias=float((p-y).mean()))
        try:
            from crvs_metrics import seg_metrics, peak_detection_scores, detect_r_peaks, hrv_from_peaks
            global_m = seg_metrics(y.reshape(-1), p.reshape(-1))
            rec.update({"val_" + k: v for k, v in global_m.items()})
            # Per-window waveform diagnostics are compact and retained for every epoch.
            den_y = np.sqrt(np.sum((y - y.mean(1, keepdims=True)) ** 2, axis=1))
            den_p = np.sqrt(np.sum((p - p.mean(1, keepdims=True)) ** 2, axis=1))
            cc = np.sum((y-y.mean(1, keepdims=True))*(p-p.mean(1, keepdims=True)), axis=1) / (den_y*den_p+1e-12)
            rec.update(val_window_CC_temporal_mean=float(100 * np.mean(cc)),
                       val_window_CC_temporal_median=float(100 * np.median(cc)),
                       val_window_CC_temporal_std=float(100 * np.std(cc)),
                       val_window_CC_temporal_p05=float(100 * np.percentile(cc, 5)),
                       val_window_MAE_mean=float(np.mean(np.abs(y-p))),
                       val_window_MSE_mean=float(np.mean((y-p)**2)))
            win = {"epoch": np.full(len(y), epoch + 1), "window": np.arange(len(y)),
                   "mae": np.mean(np.abs(y-p), 1), "mse": np.mean((y-p)**2, 1),
                   "cc_temporal": 100*cc}
            try:
                import pandas as pd
                frame = pd.DataFrame(win)
                if hasattr(val_ds, "index") and len(val_ds.index) == len(frame):
                    for col in ("rec_id", "subject", "scenario_canon", "start"):
                        if col in val_ds.index: frame[col] = val_ds.index[col].to_numpy()
                vd = self.out / "validation_windows"; vd.mkdir(exist_ok=True)
                frame.to_parquet(vd / f"epoch_{epoch+1:04d}.parquet", index=False)
            except Exception as e:
                rec["val_window_table_error"] = f"{type(e).__name__}: {e}"
            # Never concatenate different recordings: that fabricates a beat interval at
            # each boundary. Compute peak/HRV metrics per recording, then macro-average.
            record_rows = []
            if hasattr(val_ds, "index") and len(val_ds.index) == len(y):
                ix = val_ds.index.reset_index(drop=True).assign(_row=np.arange(len(y)))
                for rid, grp in ix.groupby("rec_id"):
                    pos = grp.sort_values("start")["_row"].to_numpy()
                    if len(pos) < 2: continue
                    yg = np.concatenate(y[pos]); pg = np.concatenate(p[pos])
                    peak_s = peak_detection_scores(yg, pg)
                    gt_hrv = hrv_from_peaks(detect_r_peaks(yg))
                    pr_hrv = hrv_from_peaks(detect_r_peaks(pg))
                    rr = {"rec_id": rid, "subject": str(grp["subject"].iloc[0]), **peak_s}
                    for k in ("mean_rr_ms", "sd_rr_ms", "mean_hr_bpm", "sd_hr_bpm", "rmssd_ms"):
                        rr[f"true_{k}"] = gt_hrv[k]; rr[f"pred_{k}"] = pr_hrv[k]
                        rr[f"abs_error_{k}"] = abs(pr_hrv[k]-gt_hrv[k])
                    record_rows.append(rr)
            if record_rows:
                import pandas as pd
                rdf = pd.DataFrame(record_rows)
                rd = self.out / "validation_recordings"; rd.mkdir(exist_ok=True)
                rdf.to_parquet(rd / f"epoch_{epoch+1:04d}.parquet", index=False)
                for k in ("TP", "FP", "FN", "precision", "recall", "F1", "accuracy",
                          "timing_err_ms_median", "timing_err_ms_iqr", "missed_rate"):
                    rec[f"val_wave_peak_{k}"] = float(rdf[k].mean())
                for k in ("mean_rr_ms", "sd_rr_ms", "mean_hr_bpm", "sd_hr_bpm", "rmssd_ms"):
                    rec[f"val_hrv_true_{k}"] = float(rdf[f"true_{k}"].mean())
                    rec[f"val_hrv_pred_{k}"] = float(rdf[f"pred_{k}"].mean())
                    rec[f"val_hrv_abs_error_{k}"] = float(rdf[f"abs_error_{k}"].mean())
        except Exception as e:
            rec["val_signal_metrics_error"] = f"{type(e).__name__}: {e}"
        if PH:
            ph = np.concatenate(PH); prob = 1 / (1 + np.exp(-np.clip(ph, -30, 30)))
            truth = pk >= 0.5; guess = prob >= 0.5
            tp = int(np.sum(truth & guess)); fp = int(np.sum(~truth & guess)); fn = int(np.sum(truth & ~guess))
            prec = tp / max(tp+fp, 1); recall = tp / max(tp+fn, 1)
            rec.update(val_peak_head_TP=tp, val_peak_head_FP=fp, val_peak_head_FN=fn,
                       val_peak_head_precision=prec, val_peak_head_recall=recall,
                       val_peak_head_F1=2*prec*recall/max(prec+recall, 1e-12))
        if RH:
            rr = np.concatenate(RR); rh = np.concatenate(RH); mask = rr > 0
            if np.any(mask):
                rec["val_rr_head_mae_ms"] = float(np.mean(np.abs(rr[mask]-rh[mask]))*1000)
                rec["val_rr_head_rmse_ms"] = float(np.sqrt(np.mean((rr[mask]-rh[mask])**2))*1000)
        return rec

    def _write_epoch(self, rec):
        _append_jsonl(self.out / "epoch_metrics.jsonl", rec)
        # CSV is convenient in Kaggle; JSONL remains the lossless schema-of-record.
        rows = self.state["history"]
        keys = sorted({k for r in rows for k in r})
        tmp = self.out / "epoch_metrics.csv.tmp"
        with open(tmp, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=keys); w.writeheader()
            for row in rows: w.writerow({k: _jsonable(row.get(k)) for k in keys})
            f.flush(); os.fsync(f.fileno())
        os.replace(tmp, self.out / "epoch_metrics.csv")

    def fit(self, train_ds, val_ds):
        start = int(self.state["epoch"])
        if start >= self.epochs:
            print(f"  {self.run_id} already complete at epoch {start}/{self.epochs}")
            return self.state
        if self.state.get("done"):
            self.state["done"] = False
        self._active = True
        if self.sync: self.sync.set_before_final_flush(self.emergency_checkpoint)
        run_t0 = time.time()
        try:
            for ep in range(start, self.epochs):
                if torch.cuda.is_available():
                    for gpu_i in range(torch.cuda.device_count()):
                        torch.cuda.reset_peak_memory_stats(gpu_i)
                tl = self._loader(train_ds, True, ep)
                resume_batch = int(self.state.get("batch_in_epoch", 0)) if int(self.state.get("active_epoch", ep)) == ep else 0
                partial = self.state.get("partial", {}) if resume_batch else {}
                sums = {k: float(v) for k, v in partial.get("sums", {}).items()}
                n = int(partial.get("n", 0)); samples = int(partial.get("samples", 0))
                grad_sum = float(partial.get("grad_sum", 0)); clip_events = int(partial.get("clip_events", 0))
                epoch_t0 = time.time(); data_t = 0.0; compute_t = 0.0; last_end = time.time()
                self.state.update(active_epoch=ep, batch_in_epoch=resume_batch, done=False)
                self.model.train()
                for i, batch in enumerate(tl):
                    data_t += time.time() - last_end
                    if i < resume_batch:
                        last_end = time.time(); continue
                    step_t0 = time.time(); self.opt.zero_grad(set_to_none=True)
                    loss, parts, _, _, _, _ = self._step(batch, True)
                    if not torch.isfinite(loss):
                        self.save("state", reason="non-finite-loss")
                        raise FloatingPointError(f"non-finite loss at epoch {ep+1}, batch {i+1}")
                    self.scaler.scale(loss).backward(); self.scaler.unscale_(self.opt)
                    grad = float(torch.nn.utils.clip_grad_norm_(
                        self.raw_model.parameters(), self.grad_clip or float("inf")))
                    if self.grad_clip and grad > self.grad_clip: clip_events += 1
                    self.scaler.step(self.opt); self.scaler.update()
                    compute_t += time.time() - step_t0
                    values = {"total": float(loss.detach()), **parts}
                    for k, v in values.items(): sums[k] = sums.get(k, 0.0) + float(v)
                    n += 1; samples += int(batch[0].shape[0]); grad_sum += grad
                    self.state["global_step"] = int(self.state.get("global_step", 0)) + 1
                    self.state["batch_in_epoch"] = i + 1
                    self.state["partial"] = {"sums": sums, "n": n, "samples": samples,
                                             "grad_sum": grad_sum, "clip_events": clip_events}
                    if (i + 1) % self.log_every == 0 or i + 1 == len(tl):
                        brec = {"ts": datetime.now(timezone.utc).isoformat(), "run_id": self.run_id,
                                "epoch": ep+1, "batch": i+1, "batches": len(tl),
                                "global_step": self.state["global_step"], "loss": float(loss),
                                "grad_norm": grad, "lr": self.opt.param_groups[0]["lr"],
                                "amp_scale": float(self.scaler.get_scale()),
                                "windows_per_s": int(batch[0].shape[0])/max(time.time()-step_t0, 1e-9)}
                        brec.update({"loss_"+k: v for k, v in parts.items()}); self._write_batch(brec)
                    due_step = self.state["global_step"] % self.checkpoint_every_steps == 0
                    due_time = time.time() - self._last_checkpoint >= self.checkpoint_every_s
                    if due_step or due_time:
                        self.save("state", reason="mid-epoch")
                    last_end = time.time()
                if n == 0: raise RuntimeError("training loader yielded zero batches")
                self.sched.step(); val = self._validation(val_ds, ep)
                rec = {"ts": datetime.now(timezone.utc).isoformat(), "run_id": self.run_id,
                       "epoch": ep+1, "epochs_planned": self.epochs,
                       "global_step": self.state["global_step"], "train_batches": n,
                       "train_windows": samples, "train_grad_norm_mean": grad_sum/max(n,1),
                       "train_grad_clip_events": clip_events,
                       "train_data_seconds": data_t, "train_compute_seconds": compute_t,
                       "train_windows_per_s": samples/max(compute_t, 1e-9),
                       "epoch_seconds": time.time()-epoch_t0,
                       "elapsed_seconds": time.time()-run_t0,
                       "lr": self.opt.param_groups[0]["lr"],
                       "amp_scale": float(self.scaler.get_scale())}
                rec.update(_mean_parts(sums, n, "train")); rec.update(val); rec.update(_system_stats(self.out))
                with torch.no_grad():
                    rec["model_parameter_l2"] = math.sqrt(sum(
                        float(torch.sum(p.detach().float() ** 2)) for p in self.raw_model.parameters()))
                va = float(rec["val_total"])
                if self.monitor not in rec:
                    raise KeyError(f"configured monitor '{self.monitor}' is absent from validation metrics")
                monitored = float(rec[self.monitor])
                if not math.isfinite(monitored):
                    raise FloatingPointError(
                        f"non-finite monitor {self.monitor} at epoch {ep+1}: {monitored}")
                best_so_far = float(self.state["best"])
                improved = ((monitored < best_so_far - 1e-6) if self.monitor_mode == "min"
                            else (monitored > best_so_far + 1e-6))
                if improved:
                    self.state["best"] = monitored; self.state["best_epoch"] = ep+1
                    self.state["bad_epochs"] = 0
                else:
                    self.state["bad_epochs"] = int(self.state.get("bad_epochs", 0)) + 1
                rec.update(improved=bool(improved), monitor=self.monitor,
                           monitor_mode=self.monitor_mode, monitor_value=monitored,
                           best_monitor=float(self.state["best"]),
                           best_val=float(self.state["best"]),
                           best_epoch=int(self.state["best_epoch"]),
                           bad_epochs=int(self.state["bad_epochs"]))
                self.state["epoch"] = ep+1; self.state["active_epoch"] = ep+1
                self.state["batch_in_epoch"] = 0; self.state["partial"] = {}
                self.state["history"].append(rec)
                if improved: self.save("best", reason="new-best-local")
                self.save("state", reason="epoch-complete"); self._write_epoch(rec)
                if self.sync:
                    self.sync.log("epoch", run_id=self.run_id, epoch=ep+1,
                                  train_total=rec.get("train_total"), val_total=va,
                                  cc_t=rec.get("val_CC_temporal"), cc_s=rec.get("val_CC_spectral"),
                                  hr_mae_bpm=rec.get("val_hrv_abs_error_mean_hr_bpm"), improved=improved)
                eta = rec["epoch_seconds"] * max(self.epochs-ep-1, 0) / 3600
                print(f"  ep {ep+1:>3}/{self.epochs} train {rec['train_total']:.5f} "
                      f"val {va:.5f} CCt-win {rec.get('val_window_CC_temporal_mean', float('nan')):.1f} "
                      f"CCt-global {rec.get('val_CC_temporal', float('nan')):.1f} "
                      f"CCs {rec.get('val_CC_spectral', float('nan')):.1f} "
                      f"{'*' if improved else ''} {rec['epoch_seconds']:.0f}s ETA {eta:.1f}h")
                if ((ep + 1) >= self.min_epochs and
                        int(self.state["bad_epochs"]) >= self.patience):
                    self.state["stop_reason"] = "early_stopping"; break
            self.state["done"] = True
            self.state["finished_utc"] = datetime.now(timezone.utc).isoformat()
            self.save("state", reason="run-complete")
            if self.sync: self.sync.log("training_complete", run_id=self.run_id)
            return self.state
        except KeyboardInterrupt:
            # SIGINT normally reaches HFSync first: its final hook already saved and pushed.
            # Avoid a duplicate commit; a directly-raised KeyboardInterrupt still takes this path.
            if time.time() - self._last_checkpoint > 2:
                self.save("state", reason="keyboard-interrupt")
            if self.sync:
                if not self.sync.recently_pushed(5):
                    self.sync.flush(final=True, force=True,
                                    msg=f"{self.run_id} stopped by user", run_final_hook=False)
            raise
        except Exception as e:
            self.state["last_error"] = f"{type(e).__name__}: {e}"
            self.save("state", reason="training-error")
            if self.sync:
                self.sync.log("training_error", run_id=self.run_id, error=self.state["last_error"])
                self.sync.flush(final=True, force=True,
                                msg=f"{self.run_id} error checkpoint", run_final_hook=False)
            raise
        finally:
            self._active = False
            if self.sync: self.sync.set_before_final_flush(None)

    @torch.no_grad()
    def predict(self, ds, max_keep=None):
        bp = self.out / "best.pt"
        if bp.exists():
            d = torch.load(bp, map_location=self.device, weights_only=False)
            self.raw_model.load_state_dict(d["model"], strict=True)
        self.model.eval(); Y = []; P = []
        for batch in self._loader(ds, False, 0):
            x, y, pk, rr = [b.to(self.device, non_blocking=True) for b in batch]
            with _autocast(self.device.type, self.amp): out = self.model(x)
            Y.append(y.squeeze(1).float().cpu().numpy())
            P.append(out["wave"].squeeze(1).float().cpu().numpy())
        Y = np.concatenate(Y); P = np.concatenate(P)
        if max_keep is not None: return Y[:max_keep], P[:max_keep]
        return Y, P
""",
}
for nm, src in MODULES.items():
    (WORK / nm).write_text(src)
import importlib
for nm in MODULES:
    sys.modules.pop(nm[:-3], None)
importlib.invalidate_caches()
import crvs_data
REQUIRED_LIB = 4
if getattr(crvs_data, "LIB_VERSION", 0) < REQUIRED_LIB:
    raise RuntimeError(f"stale crvs_data v{getattr(crvs_data,'LIB_VERSION','missing')}, "
                       f"need >= {REQUIRED_LIB}. Restart the kernel.")
print(f"library written ({len(MODULES)} modules), crvs_data v{crvs_data.LIB_VERSION}")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("\n" + "="*74 +
            "\n  HF_TOKEN not found. Add-ons -> Secrets -> HF_TOKEN (write) -> attach.\n" + "="*74)

from crvs_sync import HFSync
sync = HFSync(repo_id=CFG["RESULT_REPO"], local_dir=WORK, token=HF_TOKEN, repo_type="model",
              private=CFG["HF_PRIVATE"], run_id=CFG["RUN_ID"],
              push_interval_s=CFG["PUSH_INTERVAL_S"],
              max_upload_calls_hour=CFG["HF_MAX_UPLOADS_HOUR"])
print("\nresults repo:", sync.url)
sync.pull(allow_patterns=["*.json", "*.jsonl", "*.md", "tables/*", "figures/*"])
if CFG["REBUILD_CANONICAL_RESULTS"]:
    removed = []
    for folder, pattern in ((WORK / "tables", "*.csv"), (WORK / "figures", "*.png")):
        for path in folder.glob(pattern):
            path.unlink()
            removed.append(str(path.relative_to(WORK)))
    for name in ("RESULTS.md", "input_audit.json", "results_generation.json"):
        path = WORK / name
        if path.exists():
            path.unlink()
            removed.append(name)
    print(f"cleared {len(removed)} restored result artifact(s); rebuilding canonical outputs")
_M = {"f": False, "n": ""}
def MAJOR(nm):
    _M["f"] = True; _M["n"] = nm
def _hook(r=None):
    if _M["f"]:
        nm = _M["n"]; _M["f"] = False; _M["n"] = ""; sync.stage_done(nm)
try:
    get_ipython().events.register("post_run_cell", _hook)
except Exception:
    pass
MAJOR("00_setup")

library written (7 modules), crvs_data v4
HF_TOKEN loaded from Kaggle Secrets.
  [sync_started] repo=Shanmuk4622/cardiomamba-results-v2 private=False interval_s=1800 sync_version=2

results repo: https://huggingface.co/Shanmuk4622/cardiomamba-results-v2


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

  [resume_pull_ok] path=/kaggle/working/nb05 patterns=['*.json', '*.jsonl', '*.md', 'tables/*', 'figures/*']
cleared 18 restored result artifact(s); rebuilding canonical outputs
  [stage_done] stage=00_setup


---
# 2 · Collect every run

Pull the per-run summaries, per-window metrics and per-subject HRV from the model repo.
Checkpoints (`*.pt`) are **not** downloaded unless §7 runs — they are the bulk of the repo and the
tables do not need them.

In [4]:
from huggingface_hub import snapshot_download
pats = ["runs/**/summary.json", "runs/**/metrics_windows.parquet",
        "runs/**/metrics_subjects.parquet", "runs/**/metrics_recordings.parquet",
        "runs/**/preds_sample.npz",
        "runs/**/state.json", "runs/**/run_config.json", "results/*", "README.md",
        "sync_state.json", "last_push.json"]
t0 = time.time()
RUN_ROOTS = []
RUN_REPOS = []
for label, repo in (("baselines", CFG["BASELINE_REPO"]), ("cardiomamba", CFG["MODEL_REPO"])):
    root = SCRATCH / label
    snapshot_download(repo, repo_type="model", token=HF_TOKEN,
                      local_dir=str(root), allow_patterns=pats, max_workers=4)
    RUN_ROOTS.append(root)
    RUN_REPOS.append(repo)
print(f"downloaded in {time.time()-t0:.0f}s")

def norm_variant(s):
    v = s.get("variant") or s.get("model")
    if v == "multireslinknet" and s.get("loss", "mse") == "mse":
        return "multireslinknet"
    return v

rows, wrows, srows = [], [], []
summary_paths = []
for source_label, root in zip(("baselines", "cardiomamba"), RUN_ROOTS):
    summary_paths.extend((source_label, p) for p in (root / "runs").glob("*/summary.json"))
quick_skipped = []
for source_label, p in sorted(summary_paths, key=lambda x: str(x[1])):
    try:
        s = json.loads(p.read_text())
    except Exception:
        continue
    run_id = str(s.get("run_id", p.parent.name))
    if run_id.startswith("quick__") and not CFG["INCLUDE_QUICK_RUNS"]:
        quick_skipped.append(run_id)
        continue
    v = norm_variant(s)
    rc_path = p.parent / "run_config.json"
    try:
        rc = json.loads(rc_path.read_text()) if rc_path.exists() else {}
    except Exception:
        rc = {}
    protocol = str(s.get("protocol_id", rc.get("protocol_id", ""))).lower()
    target_01 = bool(s.get("target_01", rc.get("target_01", "target01" in protocol)))
    metric_values = {k: val for k, val in s["metrics"].items() if not k.endswith("_std")}
    # NB03 was trained/evaluated on [0,1]; NB04 uses the corpus-native [-1,1]. Convert the
    # exactly transformable error metrics to [0,1], which is also the paper's reported scale.
    # Preserve recorded values because temporal RRMSE cannot be transformed without raw y.
    if "MAE" in metric_values:
        metric_values["MAE_recorded"] = metric_values["MAE"]
    if "MSE" in metric_values:
        metric_values["MSE_recorded"] = metric_values["MSE"]
    if not target_01:
        if "MAE" in metric_values:
            metric_values["MAE"] = float(metric_values["MAE"]) / 2.0
        if "MSE" in metric_values:
            metric_values["MSE"] = float(metric_values["MSE"]) / 4.0
    base = {"run_id": run_id, "source_repo": source_label,
            "experiment": s["experiment"], "variant": v,
            "target_scale_recorded": "[0,1]" if target_01 else "[-1,1]",
            "comparison_scale": "[0,1]",
            "fold": s["fold"], "params": s.get("params"), "best_epoch": s.get("best_epoch"),
            "gflops_per_window": s.get("gflops_per_window"),
            "forward_ms": s.get("forward_ms_batch2", s.get("forward_ms_batch4"))}
    rows.append({**base, **metric_values})
    mw = p.parent / "metrics_windows.parquet"
    if mw.exists():
        d = pd.read_parquet(mw)
        if "MAE" in d:
            d["MAE_recorded"] = d["MAE"]
        if "MSE" in d:
            d["MSE_recorded"] = d["MSE"]
        if not target_01:
            if "MAE" in d:
                d["MAE"] = d["MAE"] / 2.0
            if "MSE" in d:
                d["MSE"] = d["MSE"] / 4.0
        d["target_scale_recorded"] = "[0,1]" if target_01 else "[-1,1]"
        d["comparison_scale"] = "[0,1]"
        d["variant"] = v; d["experiment"] = s["experiment"]
        d["fold"] = s["fold"]; wrows.append(d)
    msj = p.parent / "metrics_subjects.parquet"
    if msj.exists():
        d = pd.read_parquet(msj); d["variant"] = v; d["experiment"] = s["experiment"]
        d["fold"] = s["fold"]; srows.append(d)

R  = pd.DataFrame(rows)
WD = pd.concat(wrows, ignore_index=True) if wrows else pd.DataFrame()
SD = pd.concat(srows, ignore_index=True) if srows else pd.DataFrame()
if not len(R):
    raise RuntimeError("No runs found. Run NB03 and NB04 first.")
R.to_csv(WORK / "tables" / "all_runs.csv", index=False)
print(f"runs: {len(R)}   window rows: {len(WD):,}   subject rows: {len(SD):,}")
if quick_skipped:
    print(f"excluded {len(quick_skipped)} quick smoke run(s): {quick_skipped[:5]}")
print("\nruns per experiment x variant:")
print(R.pivot_table(index="variant", columns="experiment", values="fold",
                    aggfunc="count", fill_value=0).to_string())

baseline_runs = int((R["source_repo"] == "baselines").sum())
nb04_canonical = ((R["source_repo"] == "cardiomamba") &
                  ~R["run_id"].astype(str).str.startswith("quick__"))
nb04_full_runs = int(nb04_canonical.sum())
try:
    BASELINE_GATE = json.loads(
        (RUN_ROOTS[0] / "results" / "reproduction_gate.json").read_text(encoding="utf-8"))
except Exception as e:
    BASELINE_GATE = {"complete": False, "passed": False,
                     "read_error": f"{type(e).__name__}: {e}"}
CORE_VARIANTS = ["L2_loss_only", "L3_c1_only", "L4_c1_c5", "L5_no_wavelet",
                 "L6_no_ssm", "L7_singletask", "L8_no_film", "L9_full",
                 "L10_transformer"]
core_counts = (R[(R["source_repo"] == "cardiomamba") &
                 (R["experiment"] == CFG["HEADLINE_EXP"])]
               .groupby("variant").size().to_dict())
CORE_LADDER_COMPLETE = all(int(core_counts.get(v, 0)) >= 5 for v in CORE_VARIANTS)
EVALUATION_INPUTS_COMPLETE = (
    baseline_runs >= CFG["EXPECTED_BASELINE_RUNS"] and
    nb04_full_runs >= CFG["EXPECTED_NB04_FULL_RUNS"] and CORE_LADDER_COMPLETE)
EVALUATION_VALIDATED = EVALUATION_INPUTS_COMPLETE and bool(BASELINE_GATE.get("passed", False))
available_headline = set(R[R["experiment"] == CFG["HEADLINE_EXP"]]["variant"])
ROBUSTNESS_EFFECTIVE = bool(
    CFG["RUN_ROBUSTNESS"] and set(CFG["ROBUST_MODELS"]).issubset(available_headline))
INPUT_AUDIT = {
    "baseline_runs": baseline_runs, "expected_baseline_runs": CFG["EXPECTED_BASELINE_RUNS"],
    "nb04_full_runs": nb04_full_runs,
    "expected_nb04_full_runs": CFG["EXPECTED_NB04_FULL_RUNS"],
    "quick_runs_excluded": sorted(set(quick_skipped)),
    "core_ladder_counts": {v: int(core_counts.get(v, 0)) for v in CORE_VARIANTS},
    "core_ladder_complete": CORE_LADDER_COMPLETE,
    "baseline_gate_complete": bool(BASELINE_GATE.get("complete", False)),
    "baseline_gate_passed": bool(BASELINE_GATE.get("passed", False)),
    "evaluation_inputs_complete": EVALUATION_INPUTS_COMPLETE,
    "evaluation_validated": EVALUATION_VALIDATED,
    "robustness_requested": bool(CFG["RUN_ROBUSTNESS"]),
    "robustness_effective": ROBUSTNESS_EFFECTIVE,
    "mae_mse_comparison_scale": "[0,1]",
    "rrmse_temporal_note": "retained as recorded; target-offset dependent across NB03/NB04",
}
(WORK / "input_audit.json").write_text(json.dumps(INPUT_AUDIT, indent=2), encoding="utf-8")
print("\ninput audit:")
print(json.dumps(INPUT_AUDIT, indent=2))
if not EVALUATION_INPUTS_COMPLETE:
    print("\n>>> PARTIAL EVALUATION: NB04 full training is not complete.")
    print("    Tables remain usable for completed runs, but missing sections are not final results.")
if not BASELINE_GATE.get("passed", False):
    print(">>> NOT VALIDATED: NB03 completed, but its reproduction gate failed.")
if CFG["RUN_ROBUSTNESS"] and not ROBUSTNESS_EFFECTIVE:
    print(">>> Robustness inference will be skipped until every requested trained model exists.")
print(">>> MAE/MSE are standardized to [0,1]; *_recorded columns preserve repository values.")
print(">>> Temporal RRMSE is retained as recorded and is not cross-scale comparable.")

  [push_ok] n=1 commit=70e8338bf0a168b5e6c8924fbc3ccf228a34cf0c msg=nb05_evaluation_v2 major-stage @ 13:27Z


Fetching ... files: 0it [00:00, ?it/s]

Fetching ... files: 0it [00:00, ?it/s]

downloaded in 158s
runs: 180   window rows: 144,237   subject rows: 989
excluded 1 quick smoke run(s): ['quick__B_rva__L9_full__f0']

runs per experiment x variant:
experiment       A_apnea  A_resting  A_valsalva  B_rva  C_all5  D_loso  F_cross:Apnea  F_cross:Resting  F_cross:Tilt-down  F_cross:Tilt-up  F_cross:Valsalva
variant                                                                                                                                                     
L10_transformer        0          0           0      5       0       0              0                0                  0                0                 0
L2_loss_only           0          0           0      5       0       0              0                0                  0                0                 0
L3_c1_only             0          0           0      5       0       0              0                0                  0                0                 0
L4_c1_c5               0          0           0   

---
# 3 · Tables 2 and 3 — the direct comparison

Same layout as the baseline's tables so a reader can put them side by side. The `_paper` columns
are their published figures; `Δ` is ours minus theirs.

A reminder on how to read the sign. Our splits are strictly subject-wise with non-overlapping test
windows, which theirs almost certainly were not. So a **baseline** row landing below its published
value is expected — that is leakage being removed. The claim we are making is about
**CardioMamba-Net versus our own re-run baselines**, under identical conditions. The published
column is context, not the yardstick.

In [5]:
COLS  = ["MAE", "MSE", "CC_temporal", "CC_spectral", "RRMSE_temporal", "RRMSE_spectral"]
ORDER = ["fpn", "unet", "linknet", "multireslinknet",
         "L2_loss_only", "L3_c1_only", "L4_c1_c5", "L5_no_wavelet", "L6_no_ssm",
         "L7_singletask", "L8_no_film", "L9_full", "L10_transformer"]
NICE = {"fpn": "FPN", "unet": "UNet", "linknet": "LinkNet",
        "multireslinknet": "MultiResLinkNet", "L9_full": "CardioMamba-Net (ours)",
        "L10_transformer": "CardioMamba-Net (Transformer)"}

PAPER_A = {
 ("A_resting","fpn"):(0.14204,0.03170,58.37,71.38,0.46940,0.73374),
 ("A_resting","unet"):(0.13872,0.03219,63.10,74.68,0.45760,0.86096),
 ("A_resting","linknet"):(0.13588,0.03034,64.35,74.37,0.45116,0.81111),
 ("A_resting","multireslinknet"):(0.13258,0.03066,66.10,82.44,0.43682,0.71412),
 ("A_valsalva","fpn"):(0.14985,0.03679,57.53,65.97,0.46395,0.87990),
 ("A_valsalva","unet"):(0.15249,0.03928,58.38,68.79,0.46553,0.99554),
 ("A_valsalva","linknet"):(0.15087,0.03869,56.63,66.87,0.46195,0.99095),
 ("A_valsalva","multireslinknet"):(0.15286,0.04012,60.14,77.05,0.46083,0.80660),
 ("A_apnea","fpn"):(0.15310,0.03853,39.12,51.26,0.51017,1.00889),
 ("A_apnea","unet"):(0.14406,0.03495,56.14,69.97,0.47825,0.92034),
 ("A_apnea","linknet"):(0.14572,0.03526,56.22,70.35,0.47944,0.91749),
 ("A_apnea","multireslinknet"):(0.14474,0.03474,55.33,74.66,0.47692,0.82392),
 ("B_rva","fpn"):(0.14316,0.03422,59.63,69.53,0.44694,0.83026),
 ("B_rva","unet"):(0.14798,0.03741,57.65,68.39,0.45315,0.94118),
 ("B_rva","linknet"):(0.14780,0.03723,58.69,70.91,0.45487,0.86909),
 ("B_rva","multireslinknet"):(0.14841,0.03793,61.86,79.96,0.44618,0.73269),
}

def table_for(exps, fname, title):
    sub = R[R["experiment"].isin(exps)]
    if not len(sub):
        print(f"(no runs for {exps})"); return None
    g = sub.groupby(["experiment", "variant"])
    T = g[COLS].mean().round(5)
    Tsd = g[COLS].std().round(5)
    T["folds"] = g.size()
    T = T.reset_index()
    for i, r in T.iterrows():
        key = (r["experiment"], r["variant"])
        if key in PAPER_A:
            p = PAPER_A[key]
            T.loc[i, "CC_t_paper"] = p[2]; T.loc[i, "CC_s_paper"] = p[3]
            T.loc[i, "MAE_paper"] = p[0]
            T.loc[i, "dCC_t"] = round(r["CC_temporal"] - p[2], 2)
    T["order"] = T["variant"].map(lambda v: ORDER.index(v) if v in ORDER else 99)
    T = T.sort_values(["experiment", "order"]).drop(columns="order")
    T["variant"] = T["variant"].map(lambda v: NICE.get(v, v))
    print("=" * 130); print(title); print("=" * 130)
    print(T.to_string(index=False))
    T.to_csv(WORK / "tables" / fname, index=False)
    return T

T2 = table_for(["A_resting", "A_valsalva", "A_apnea"], "table2_per_scenario.csv",
               "TABLE 2  —  per scenario  (their Table 2)")
print()
T3 = table_for(["B_rva"], "table3_rva_combined.csv",
               "TABLE 3  —  Resting + Valsalva + Apnea combined  (their Table 3)")
print()
TC = table_for(["C_all5"], "table3b_all_five.csv",
               "TABLE 3b  —  ALL FIVE SCENARIOS  (new: the baseline never evaluated Tilt)")
print()
TD = table_for(["D_loso"], "table3c_loso.csv",
               "TABLE 3c  —  LEAVE-ONE-SUBJECT-OUT GENERALISATION")
print()
cross_exps = sorted(x for x in R["experiment"].unique() if str(x).startswith("F_cross:"))
TF = (table_for(cross_exps, "table3d_cross_scenario.csv",
                "TABLE 3d  —  HELD-OUT-SCENARIO GENERALISATION")
      if cross_exps else None)
MAJOR("01_tables_2_3")

TABLE 2  —  per scenario  (their Table 2)
experiment                variant     MAE     MSE  CC_temporal  CC_spectral  RRMSE_temporal  RRMSE_spectral  folds  CC_t_paper  CC_s_paper  MAE_paper  dCC_t
   A_apnea                    FPN 0.15469 0.03711     39.81532     75.63118         0.52456         0.78261      5       39.12       51.26    0.15310   0.70
   A_apnea                   UNet 0.15939 0.03990     20.65021     44.47213         0.55189         0.92124      5       56.14       69.97    0.14406 -35.49
   A_apnea                LinkNet 0.15062 0.03630     36.72704     68.84624         0.51611         0.85030      5       56.22       70.35    0.14572 -19.49
   A_apnea        MultiResLinkNet 0.15277 0.03657     41.75307     77.12377         0.52132         0.75752      5       55.33       74.66    0.14474 -13.58
   A_apnea CardioMamba-Net (ours) 0.16453 0.04230     45.07136     82.55568         0.92660         0.99980      5         NaN         NaN        NaN    NaN
 A_resting      

---
# 4 · Tables 4 and 5 — beats and rhythm

Table 4 is R-peak detection on the reconstructed ECG. We add two columns the baseline does not
report: **median timing error in milliseconds** and **missed-detection rate**. Precision and recall
alone hide whether a "detected" beat is 10 ms or 90 ms off, and for HRV that difference is
everything.

Table 5 is the HRV comparison, and it is where the baseline's weakness is most visible. Their
predicted RMSSD is roughly **double** ground truth in every scenario (12.95 → 23.87 ms resting;
13.17 → 31.50 ms apnea) — the fingerprint of a smeared, jittery QRS. Our RMSSD error column is
the direct test of whether C5 fixed it.

We also report μRR in **genuine milliseconds**. Theirs is 126 ms alongside a heart rate of 62 bpm,
which is arithmetically impossible — 126 samples at 128 Hz is 0.98 s, so their column is samples
mislabelled as milliseconds.

In [6]:
if len(SD):
    keep = [v for v in ORDER if v in set(SD["variant"])]
    g = SD[SD["experiment"] == CFG["HEADLINE_EXP"]].groupby("variant")
    T4 = g.agg(accuracy=("accuracy", "mean"), F1=("F1", "mean"),
               precision=("precision", "mean"), recall=("recall", "mean"),
               TP=("TP", "sum"), FP=("FP", "sum"), FN=("FN", "sum"),
               timing_err_ms=("timing_err_ms_median", "mean"),
               missed_rate=("missed_rate", "mean")).round(4)
    T4 = T4.reindex([v for v in keep if v in T4.index])
    T4.index = [NICE.get(i, i) for i in T4.index]
    print("=" * 118)
    print(f"TABLE 4  —  R-peak detection on the reconstructed ECG  ({CFG['HEADLINE_EXP']})")
    print("=" * 118)
    print(T4.to_string())
    print("\npublished (MultiResLinkNet, RVA): accuracy 0.886  F1 0.939  precision 0.973  recall 0.908")
    T4.to_csv(WORK / "tables" / "table4_peak_detection.csv")

    rows5 = []
    for v in keep:
        d = SD[(SD["variant"] == v) & (SD["experiment"] == CFG["HEADLINE_EXP"])]
        if not len(d):
            continue
        rows5.append({"variant": NICE.get(v, v), "signal": "ground truth",
                      "mu_RR_ms": d["gt_mean_rr_ms"].mean(), "sd_RR_ms": d["gt_sd_rr_ms"].mean(),
                      "mu_HR_bpm": d["gt_mean_hr_bpm"].mean(), "sd_HR_bpm": d["gt_sd_hr_bpm"].mean(),
                      "RMSSD_ms": d["gt_rmssd_ms"].mean()})
        rows5.append({"variant": NICE.get(v, v), "signal": "predicted",
                      "mu_RR_ms": d["pr_mean_rr_ms"].mean(), "sd_RR_ms": d["pr_sd_rr_ms"].mean(),
                      "mu_HR_bpm": d["pr_mean_hr_bpm"].mean(), "sd_HR_bpm": d["pr_sd_hr_bpm"].mean(),
                      "RMSSD_ms": d["pr_rmssd_ms"].mean()})
        rows5.append({"variant": NICE.get(v, v), "signal": "|error|",
                      "mu_RR_ms": (d["gt_mean_rr_ms"] - d["pr_mean_rr_ms"]).abs().mean(),
                      "sd_RR_ms": np.nan,
                      "mu_HR_bpm": (d["gt_mean_hr_bpm"] - d["pr_mean_hr_bpm"]).abs().mean(),
                      "sd_HR_bpm": np.nan,
                      "RMSSD_ms": (d["gt_rmssd_ms"] - d["pr_rmssd_ms"]).abs().mean()})
    T5 = pd.DataFrame(rows5).round(2)
    print("\n" + "=" * 118)
    print(f"TABLE 5  —  HR and HRV, in REAL milliseconds  ({CFG['HEADLINE_EXP']})")
    print("=" * 118)
    print(T5.to_string(index=False))
    T5.to_csv(WORK / "tables" / "table5_hrv.csv", index=False)
    print("\npublished (MultiResLinkNet, resting): RMSSD ground truth 12.95 ms -> predicted 23.87 ms")
    print("i.e. an 84 % over-estimate. The |error| rows above are the direct comparison.")
else:
    print("no per-subject metrics found -- NB03/NB04 write these; re-run them.")
MAJOR("02_tables_4_5")

TABLE 4  —  R-peak detection on the reconstructed ECG  (B_rva)
                               accuracy      F1  precision  recall          TP          FP          FN  timing_err_ms  missed_rate
FPN                              0.3960  0.4782     0.5804  0.4181   8325.8333   5500.6667  11885.8333        30.7726       0.5819
UNet                             0.2691  0.3311     0.4636  0.2871   5528.3333   3840.3333  14683.3333        35.4601       0.7129
LinkNet                          0.2230  0.3045     0.4048  0.2550   5259.6667  10354.0000  14952.0000        39.8003       0.7450
MultiResLinkNet                  0.5140  0.6270     0.7830  0.5545  11096.3333   3230.1667   9115.3333        22.9601       0.4455
L2_loss_only                     0.7662  0.8514     0.8810  0.8332  16644.3333   1731.8333   3567.3333        27.3872       0.1668
L3_c1_only                       0.6786  0.7883     0.9324  0.7024  13881.8333    770.8333   6329.8333        14.4965       0.2976
L4_c1_c5            

---
# 5 · Table 6 — the ablation, and Table 7 — significance

Table 7 is what lets us write "significantly better" when the data support it. Wilcoxon
signed-rank is paired and non-parametric: each subject contributes one average temporal
correlation per model, regardless of how many windows that subject has.

The planned comparisons are the full model against each baseline/ablation. Holm correction
controls the family-wise error rate without pretending that all 45 possible pairs were hypotheses
we intended to test.

In [7]:
from scipy import stats as sstats

b = R[R["experiment"] == CFG["HEADLINE_EXP"]]
T6, T7, T8 = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
LAB6 = {"multireslinknet": "1. MultiResLinkNet + MSE (baseline)",
        "L2_loss_only": "2. + composite loss (C5)",
        "L3_c1_only": "3. + 8-channel input (C1)",
        "L4_c1_c5": "4. + C1 + C5",
        "L5_no_wavelet": "5. CardioMamba, no wavelet (-C2)",
        "L6_no_ssm": "6. CardioMamba, no SSM (-C3)",
        "L7_singletask": "7. CardioMamba, single-task (-C4)",
        "L8_no_film": "8. CardioMamba, no FiLM",
        "L9_full": "9. CardioMamba-Net (full)",
        "L10_transformer": "10. Transformer bottleneck (control)"}
lad = [v for v in LAB6 if v in set(b["variant"])]
if lad:
    extra = [c for c in ("peak_F1", "MAE_mean_hr_bpm", "MAE_rmssd_ms") if c in b.columns]
    T6 = b[b["variant"].isin(lad)].groupby("variant")[COLS + extra + ["params"]].mean()
    T6 = T6.reindex(lad)
    T6["folds"] = b.groupby("variant").size().reindex(lad)
    ref = T6.loc["multireslinknet", "CC_temporal"] if "multireslinknet" in T6.index else np.nan
    T6["dCC_t"] = (T6["CC_temporal"] - ref).round(2)
    T6.index = [LAB6[i] for i in T6.index]
    print("=" * 132); print("TABLE 6  —  ablation ladder"); print("=" * 132)
    print(T6.round(5).to_string())
    T6.to_csv(WORK / "tables" / "table6_ablation.csv")

    # Subject is the independent unit. First average windows within subject/model, then
    # align the exact same subjects for every full-vs-comparator test.
    subject_cc = (WD[WD["experiment"] == CFG["HEADLINE_EXP"]]
                  .groupby(["subject", "variant"], as_index=False)["CC_temporal"].mean())
    full = "L9_full"; raw = []
    if full in set(subject_cc["variant"]):
        for v in [x for x in lad if x != full]:
            pair = subject_cc[subject_cc["variant"].isin([full, v])].pivot(
                index="subject", columns="variant", values="CC_temporal").dropna()
            if len(pair) < 6 or np.allclose(pair[full], pair[v]):
                p = np.nan
            else:
                p = float(sstats.wilcoxon(pair[full], pair[v], alternative="two-sided").pvalue)
            raw.append({"a": full, "b": v, "n_subjects": len(pair), "p": p,
                        "median_delta_cc_t": float(np.median(pair[full]-pair[v])) if len(pair) else np.nan})
    if raw:
        finite = sorted([i for i, r in enumerate(raw) if np.isfinite(r["p"])], key=lambda i: raw[i]["p"])
        running = 0.0; mtests = len(finite)
        for rank, i in enumerate(finite):
            running = max(running, (mtests-rank)*raw[i]["p"])
            raw[i]["p_holm"] = min(1.0, running)
        for r in raw: r.setdefault("p_holm", np.nan)
        T7 = pd.DataFrame(raw)
        T7["a"] = T7["a"].map(LAB6); T7["b"] = T7["b"].map(LAB6)
        T7["significant"] = T7["p_holm"] < CFG["ALPHA"]
        T7 = T7.sort_values("p_holm", na_position="last")
        print("\n" + "=" * 118)
        print("TABLE 7  —  subject-paired Wilcoxon on CC_temporal, Holm-corrected")
        print("=" * 118)
        print(T7.to_string(index=False))
        T7.to_csv(WORK / "tables" / "table7_significance.csv", index=False)
        vs = T7[(T7["a"].str.contains("full")) | (T7["b"].str.contains("full"))]
        if len(vs):
            print(f"\n  comparisons involving the full model: "
                  f"{int(vs['significant'].sum())}/{len(vs)} significant at alpha={CFG['ALPHA']}")
    else:
        print("\nTABLE 7 skipped: full-model and comparator per-subject rows are incomplete.")
        print("Run NB03/NB04 with QUICK=False so all subject-held-out folds complete.")
        pd.DataFrame(columns=["a", "b", "n_subjects", "p", "median_delta_cc_t",
                              "p_holm", "significant"]).to_csv(
            WORK / "tables" / "table7_significance.csv", index=False)

if "params" in R.columns and R["params"].notna().any():
    T8 = (R[R["experiment"] == CFG["HEADLINE_EXP"]]
          .groupby("variant").agg(params=("params", "mean"),
                                  gflops=("gflops_per_window", "mean"),
                                  forward_ms=("forward_ms", "mean"),
                                  CC_temporal=("CC_temporal", "mean")).dropna(subset=["params", "CC_temporal"]))
    T8["M_params"] = (T8["params"] / 1e6).round(3)
    T8["CC_per_Mparam"] = (T8["CC_temporal"] / T8["M_params"]).round(2)
    T8 = T8.sort_values("CC_temporal", ascending=False)
    T8.index = [NICE.get(i, LAB6.get(i, i)) for i in T8.index]
    print("\n" + "=" * 96)
    print("TABLE 8  —  budget.  The baseline paper reports neither parameters nor FLOPs.")
    print("=" * 96)
    print(T8[["M_params", "gflops", "forward_ms", "CC_temporal", "CC_per_Mparam"]].to_string())
    T8.to_csv(WORK / "tables" / "table8_budget.csv")
MAJOR("03_tables_6_7_8")

TABLE 6  —  ablation ladder
                                          MAE      MSE  CC_temporal  CC_spectral  RRMSE_temporal  RRMSE_spectral  peak_F1  MAE_mean_hr_bpm  MAE_rmssd_ms     params  folds  dCC_t
1. MultiResLinkNet + MSE (baseline)   0.16573  0.04269     44.61144     73.70150         0.53450         0.76230  0.62090         14.28233     263.95063  9218556.0      5   0.00
2. + composite loss (C5)              0.15651  0.03792     52.82052     87.83619         0.87563         0.69332  0.84510          4.06323     185.82743  9218556.0      5   8.21
3. + 8-channel input (C1)             0.16247  0.04046     59.62263     87.55379         0.89007         0.65083  0.77828          5.97986     253.34225  9221692.0      5  15.01
4. + C1 + C5                          0.15422  0.03765     56.43640     87.71232         0.86209         0.77744  0.73822          5.81239     263.38576  9221692.0      5  11.82
5. CardioMamba, no wavelet (-C2)      0.16201  0.03981     56.90780     89.09919  

---
# 6 · Figures

Ten figures, all written to `figures/` at 160 dpi and pushed. The house palette matches NB01–NB04
so the whole paper reads as one piece of work: **teal for radar (input), red for ECG (output)** —
the two accents mean something rather than decorating.

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from crvs_metrics import bland_altman

S = {"radar": "#0F7C82", "ecg": "#AF3A2C", "muted": "#5C6B71", "ink": "#10171B",
     "grid": "#D3DADB", "amber": "#8A6212", "soft": "#9BB8BA"}
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 160, "savefig.bbox": "tight",
                     "axes.grid": True, "grid.color": S["grid"], "grid.linewidth": .6,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 8.5,
                     "axes.titlesize": 10, "axes.titleweight": "bold", "legend.fontsize": 7.5})
FIG = WORK / "figures"
def save(f, nm):
    f.savefig(FIG / nm); plt.close(f); print("  wrote", nm)

# --- F1 ours vs published --------------------------------------------------
if T3 is not None and len(T3):
    d = T3.dropna(subset=["CC_t_paper"])
    if len(d):
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
        for ax, ours, paper, ttl in [(axes[0], "CC_temporal", "CC_t_paper", "temporal correlation"),
                                     (axes[1], "CC_spectral", "CC_s_paper", "spectral correlation")]:
            xs = np.arange(len(d))
            ax.bar(xs - .2, d[ours], .4, color=S["radar"], edgecolor="white", label="our run")
            ax.bar(xs + .2, d[paper], .4, color=S["muted"], edgecolor="white", label="published")
            ax.set_xticks(xs); ax.set_xticklabels(d["variant"], rotation=20, ha="right", fontsize=7)
            ax.set_title(ttl + "  (RVA combined)")
            ax.legend(frameon=False)
        fig.tight_layout(); save(fig, "fig01_ours_vs_published.png")

# --- F2 ablation ladder ----------------------------------------------------
if lad:
    vals = [b[b["variant"] == v]["CC_temporal"].mean() for v in lad]
    cols = [S["ecg"] if v == "L9_full" else S["muted"] if v == "multireslinknet"
            else S["radar"] for v in lad]
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.barh(range(len(lad)), vals, color=cols, edgecolor="white")
    ax.set_yticks(range(len(lad))); ax.set_yticklabels([LAB6[v] for v in lad], fontsize=7.5)
    ax.invert_yaxis()
    ax.axvline(61.86, color=S["ink"], ls="--", lw=1.2, label="published MultiResLinkNet")
    for i, v in enumerate(vals):
        ax.text(v + .3, i, f"{v:.1f}", va="center", fontsize=7)
    ax.set_xlabel("temporal correlation (x100)")
    ax.set_title("Ablation ladder — what each contribution is worth", loc="left")
    ax.legend(frameon=False)
    save(fig, "fig02_ablation.png")

# --- F3/F4 Bland-Altman ----------------------------------------------------
if len(SD):
    for met, unit, fn in [("mean_hr_bpm", "bpm", "fig03_bland_altman_hr.png"),
                          ("rmssd_ms", "ms", "fig04_bland_altman_rmssd.png")]:
        picks = [v for v in ("multireslinknet", "L9_full") if v in set(SD["variant"])]
        if not picks or f"gt_{met}" not in SD.columns:
            continue
        fig, axes = plt.subplots(1, len(picks), figsize=(5.4 * len(picks), 3.6), squeeze=False)
        for ax, v in zip(axes[0], picks):
            d = SD[(SD["variant"] == v) & (SD["experiment"] == CFG["HEADLINE_EXP"])].dropna(
                subset=[f"gt_{met}", f"pr_{met}"])
            if not len(d):
                continue
            ba = bland_altman(d[f"pr_{met}"].to_numpy(), d[f"gt_{met}"].to_numpy())
            c = S["ecg"] if v == "L9_full" else S["muted"]
            ax.scatter(ba["mean"], ba["diff"], s=22, color=c, alpha=.75, edgecolor="white", lw=.5)
            ax.axhline(ba["bias"], color=S["ink"], lw=1.2)
            ax.axhline(ba["loa_hi"], color=S["ink"], ls="--", lw=1)
            ax.axhline(ba["loa_lo"], color=S["ink"], ls="--", lw=1)
            ax.axhline(0, color=S["grid"], lw=.8)
            ax.set_title(f"{NICE.get(v, LAB6.get(v, v))}\nbias {ba['bias']:+.2f}  "
                         f"LoA [{ba['loa_lo']:+.1f}, {ba['loa_hi']:+.1f}] {unit}", fontsize=8.5)
            ax.set_xlabel(f"mean of methods ({unit})")
            ax.set_ylabel(f"predicted - true ({unit})")
        fig.suptitle(f"Bland–Altman agreement — {met.replace('_',' ')}", y=1.02,
                     fontsize=10, fontweight="bold")
        fig.tight_layout(); save(fig, fn)

# --- F5 per-subject box plots ---------------------------------------------
if len(WD):
    picks = [v for v in ORDER if v in set(WD["variant"])][:6]
    d = WD[(WD["variant"].isin(picks)) & (WD["experiment"] == CFG["HEADLINE_EXP"])]
    if len(d):
        fig, ax = plt.subplots(figsize=(11, 3.8))
        data = [d[d["variant"] == v]["CC_temporal"].dropna().to_numpy() for v in picks]
        bp = ax.boxplot(data, labels=[NICE.get(v, LAB6.get(v, v)) for v in picks],
                        patch_artist=True, showfliers=False, widths=.6)
        for patch, v in zip(bp["boxes"], picks):
            patch.set_facecolor(S["ecg"] if v == "L9_full" else S["radar"])
            patch.set_alpha(.75); patch.set_edgecolor("white")
        for m in bp["medians"]:
            m.set_color(S["ink"]); m.set_linewidth(1.4)
        ax.set_ylabel("temporal correlation per window (x100)")
        ax.set_title("Distribution across held-out windows — means hide the tail", loc="left")
        ax.tick_params(axis="x", rotation=14, labelsize=7)
        save(fig, "fig05_distribution.png")

    dsub = d.groupby(["variant", "subject"])["CC_temporal"].mean().reset_index()
    if len(dsub):
        fig, ax = plt.subplots(figsize=(11, 3.6))
        for k, v in enumerate(picks):
            s = dsub[dsub["variant"] == v]
            ax.scatter(np.full(len(s), k) + np.random.uniform(-.14, .14, len(s)),
                       s["CC_temporal"], s=26,
                       color=S["ecg"] if v == "L9_full" else S["radar"],
                       alpha=.8, edgecolor="white", lw=.5)
        ax.set_xticks(range(len(picks)))
        ax.set_xticklabels([NICE.get(v, LAB6.get(v, v)) for v in picks], rotation=14, fontsize=7)
        ax.set_ylabel("per-subject mean CC_temporal (x100)")
        ax.set_title("Per-subject performance — one dot per held-out subject", loc="left")
        save(fig, "fig06_per_subject.png")

# --- F7 qualitative grid ---------------------------------------------------
sp = {}
for root in RUN_ROOTS:
    for p in sorted((root / "runs").glob("*/preds_sample.npz")):
        nm = p.parent.name
        is_quick = nm.startswith("quick__")
        if is_quick and not CFG["INCLUDE_QUICK_RUNS"]:
            continue
        logical_nm = nm[len("quick__"):] if is_quick else nm
        for v in ORDER:
            if (f"__{v}__" in logical_nm and
                    logical_nm.startswith(CFG["HEADLINE_EXP"]) and v not in sp):
                sp[v] = p
picks = [v for v in ("multireslinknet", "L6_no_ssm", "L9_full") if v in sp]
if picks:
    z0 = np.load(sp[picks[0]])
    k = min(5, len(z0["y"]) - 1); tt = np.arange(1024) / 128.0
    fig, axes = plt.subplots(len(picks) + 1, 1, figsize=(11, 1.9 * (len(picks) + 1)), sharex=True)
    axes[0].plot(tt, z0["y"][k], lw=1.2, color=S["ink"])
    axes[0].set_ylabel("ground truth", rotation=0, ha="right", va="center", fontsize=8)
    for ax, v in zip(axes[1:], picks):
        z = np.load(sp[v]); kk = min(k, len(z["p"]) - 1)
        ax.plot(tt, z["p"][kk], lw=1.2,
                color=S["ecg"] if v == "L9_full" else S["muted"])
        ax.set_ylabel(NICE.get(v, LAB6.get(v, v)), rotation=0, ha="right", va="center", fontsize=7.5)
    for a in axes:
        a.tick_params(labelleft=False)
    axes[-1].set_xlabel("seconds")
    axes[0].set_title("Held-out reconstruction — the question is whether the QRS stays sharp",
                      loc="left")
    fig.tight_layout(); save(fig, "fig07_qualitative.png")

# --- F8 budget scatter -----------------------------------------------------
if "params" in R.columns and R["params"].notna().any():
    d = (R[R["experiment"] == CFG["HEADLINE_EXP"]]
         .groupby("variant").agg(p=("params", "mean"), c=("CC_temporal", "mean")).dropna())
    if len(d):
        fig, ax = plt.subplots(figsize=(7.6, 4.2))
        for v, r in d.iterrows():
            col = S["ecg"] if v == "L9_full" else S["muted"] if v in NICE else S["radar"]
            ax.scatter(r["p"] / 1e6, r["c"], s=110, color=col, edgecolor="white", lw=1, zorder=3)
            ax.annotate(NICE.get(v, LAB6.get(v, v)).split(".")[-1].strip(),
                        (r["p"] / 1e6, r["c"]), fontsize=6.8, xytext=(5, 4),
                        textcoords="offset points")
        ax.axhline(61.86, color=S["ink"], ls="--", lw=1, label="published MultiResLinkNet")
        ax.set_xlabel("parameters (millions)"); ax.set_ylabel("CC_temporal (x100)")
        ax.set_title("Accuracy against model size — smaller and better is the claim", loc="left")
        ax.legend(frameon=False)
        save(fig, "fig08_budget.png")
MAJOR("04_figures")

  [push_ok] n=2 commit=cb7275e03b6afd2c335f9f83534c53755fe22a57 msg=nb05_evaluation_v2 major-stage @ 13:30Z
  wrote fig01_ours_vs_published.png
  wrote fig02_ablation.png
  wrote fig03_bland_altman_hr.png
  wrote fig04_bland_altman_rmssd.png
  wrote fig05_distribution.png
  wrote fig06_per_subject.png
  wrote fig07_qualitative.png
  wrote fig08_budget.png
  [stage_done] stage=04_figures


---
# 7 · Experiment E — robustness *(optional, needs GPU)*

The baseline never tests robustness. radarODE-MTL set the precedent that it matters, and a reviewer
will ask: what happens when the radar signal is noisier than a clinical recording room?

We test white noise across SNRs, low-frequency motion drift, and every single-channel dropout, then
retain all waveform metrics for every condition. No retraining — this measures how gracefully
each model degrades and which physics channel it relies on. Set
`CFG["RUN_ROBUSTNESS"] = False` to skip.

In [9]:
if not ROBUSTNESS_EFFECTIVE:
    if not CFG["RUN_ROBUSTNESS"]:
        print("robustness skipped (CFG['RUN_ROBUSTNESS'] = False)")
    else:
        missing = sorted(set(CFG["ROBUST_MODELS"]) - available_headline)
        print("robustness skipped: trained canonical checkpoint(s) missing for", missing)
        print("quick smoke checkpoints are intentionally not treated as final models.")
    ROB = pd.DataFrame(columns=["variant", "corruption", "level", "channel", "n",
                                "comparison_scale", "CC_temporal", "CC_spectral",
                                "MAE", "MSE", "RRMSE_temporal", "RRMSE_spectral"])
    # Always replace a table from an earlier/partial NB05 run; an empty schema is explicit.
    ROB.to_csv(WORK / "tables" / "table9_robustness.csv", index=False)
else:
    import torch
    from huggingface_hub import hf_hub_download
    from crvs_data import WindowDataset, FS
    from crvs_models import build_baseline
    from crvs_cmnet import build_cmnet
    from crvs_metrics import seg_metrics
    from crvs_engine import pick_device, seed_all

    dev, ngpu, _ = pick_device()
    print("device:", dev, "| gpus:", ngpu)
    if dev.type != "cuda":
        print("  (CPU -- this will be slow; consider setting RUN_ROBUSTNESS=False)")

    DATA = None
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for candidate in input_root.rglob("windows.parquet"):
            if (candidate.parent / "recordings").exists() and (candidate.parent / "norm_stats.json").exists():
                DATA = candidate.parent; break
    if DATA is None:
        DATA = SCRATCH / "corpus"
        snapshot_download(CFG["DATA_REPO"], repo_type="dataset", token=HF_TOKEN,
                          local_dir=str(DATA),
                          allow_patterns=["recordings/*.npy", "recordings/*.json",
                                          "recordings/*.npz", "windows.parquet", "norm_stats.json",
                                          "experiments.json"], max_workers=4)
    else:
        print("using attached Kaggle NB02 output:", DATA)
    Wn = pd.read_parquet(DATA / "windows.parquet")
    NORM = json.loads((DATA / "norm_stats.json").read_text())
    EXPINFO = json.loads((DATA / "experiments.json").read_text())
    EXPERIMENTS = EXPINFO["experiments"]
    REC_DIR = DATA / "recordings"

    class BaselineOutputConvention(torch.nn.Module):
        """Recreate NB03's recorded [0,1] target/output convention exactly."""
        def __init__(self, base, target_01=False):
            super().__init__()
            self.base = base
            self.target_01 = bool(target_01)
        def forward(self, x):
            pred = self.base(x)
            if self.target_01:
                pred["wave"] = (pred["wave"] + 1.0) * 0.5
                if "aux" in pred:
                    pred["aux"] = [torch.sigmoid(v) for v in pred["aux"]]
            return pred

    def load_run(run_dir):
        s = json.loads((run_dir / "summary.json").read_text())
        rc_path = run_dir / "run_config.json"
        rc = json.loads(rc_path.read_text()) if rc_path.exists() else {}
        v = norm_variant(s)
        spec = s.get("spec", {})
        ch = spec.get("channels") or s.get("channels") or ["dy"]
        sd = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)["model"]
        target_01 = False
        is_cmnet = (spec.get("kind") == "cmnet" or
                    ((v or "").startswith("L") and spec.get("kind") != "baseline"))
        if is_cmnet:
            m = build_cmnet(in_ch=len(ch), base=rc.get("base", 32),
                            levels=rc.get("levels", 4), d_ssm=rc.get("d_ssm", 256),
                            ssm_blocks=rc.get("ssm_blocks", 3), d_state=rc.get("d_state", 64),
                            bottleneck=spec.get("bottleneck", "ssm"),
                            use_wavelet=spec.get("wavelet", True),
                            multitask=spec.get("multitask", True),
                            use_film=spec.get("film", True), dropout=rc.get("dropout", 0.1))
            m.load_state_dict(sd, strict=True)
        else:
            base_model = build_baseline(spec.get("model", s.get("model", "multireslinknet")),
                                        in_ch=len(ch), out_ch=1, base=rc.get("base", 64),
                                        levels=rc.get("levels", 4))
            target_01 = bool(rc.get("target_01", s.get("target_01", False)))
            if any(str(k).startswith("base.") for k in sd):
                m = BaselineOutputConvention(base_model, target_01)
                m.load_state_dict(sd, strict=True)
            else:
                base_model.load_state_dict(sd, strict=True)
                m = BaselineOutputConvention(base_model, target_01) if target_01 else base_model
        return m.to(dev).eval(), ch, s, v, target_01

    rob_path = WORK / "tables" / "table9_robustness.csv"
    rob = pd.read_csv(rob_path).to_dict("records") if rob_path.exists() else []
    finished = {(str(r["variant"]), str(r["corruption"]), f"{float(r.get('level', 0)):g}",
                 "" if pd.isna(r.get("channel", "")) else str(r.get("channel", ""))) for r in rob}
    seed_all(CFG["SEED"])
    for v in CFG["ROBUST_MODELS"]:
        cand = []
        for root, repo in zip(RUN_ROOTS, RUN_REPOS):
            prefixes = [""] + (["quick__"] if CFG["INCLUDE_QUICK_RUNS"] else [])
            for prefix in prefixes:
                cand.extend((p, repo, root) for p in (root / "runs").glob(
                    f"{prefix}{CFG['HEADLINE_EXP']}__{v}__f*")
                    if (p / "summary.json").exists())
        if not cand:
            print(f"  no checkpoint for {v} -- skipped"); continue
        rd, source_repo, source_root = sorted(cand, key=lambda x: str(x[0]))[0]
        if not (rd / "best.pt").exists():
            print(f"  downloading one selected checkpoint: {rd.name}/best.pt")
            hf_hub_download(source_repo, f"runs/{rd.name}/best.pt", repo_type="model",
                            token=HF_TOKEN, local_dir=str(source_root))
        try:
            model, ch, s, vv, target_01 = load_run(rd)
        except Exception as e:
            print(f"  could not load {rd.name}: {type(e).__name__}: {e}"); continue
        fold = s["fold"]
        sub = Wn[Wn["scenario_canon"].isin(EXPERIMENTS[CFG["HEADLINE_EXP"]])]
        te = sub[(sub["fold_group"] == fold % 5) & sub["no_overlap"]]
        te = te.iloc[:CFG["ROBUST_MAX_WINDOWS"]]
        norm = NORM[f"{CFG['HEADLINE_EXP']}|{fold}"]
        idx = [EXPINFO["channels"].index(c) for c in ch]
        sn = {"mean": [norm["mean"][i] for i in idx], "std": [norm["std"][i] for i in idx]}
        ds = WindowDataset(REC_DIR, te, sn, ch, augment=False)
        print(f"\n  {v}: {len(ds)} test windows, {len(ch)} channel(s)")
        cases = [("clean", 0, "")] + [("awgn", x, "") for x in CFG["SNR_DB"]]
        cases += [("motion_drift", x, "") for x in CFG["MOTION_AMPLITUDE"]]
        if CFG["TEST_CHANNEL_DROPOUT"]:
            cases += [("channel_dropout", 1, c) for c in ch]
        for case_i, (corruption, level, channel) in enumerate(cases):
            key = (v, corruption, f"{float(level):g}", str(channel))
            if key in finished:
                print(f"    {corruption} {level} {channel} restored"); continue
            seed_all(CFG["SEED"] + 1000 * CFG["ROBUST_MODELS"].index(v) + case_i)
            mets = []
            with torch.no_grad():
                for i in range(0, len(ds), 32):
                    xb, yb = [], []
                    for j in range(i, min(i + 32, len(ds))):
                        x, y, _, _ = ds[j]; xb.append(x); yb.append(y)
                    X = torch.stack(xb).to(dev); Y = torch.stack(yb)
                    if corruption == "awgn":
                        p_sig = X.pow(2).mean(dim=(1, 2), keepdim=True)
                        X = X + torch.randn_like(X) * (p_sig / (10 ** (float(level)/10))).sqrt()
                    elif corruption == "motion_drift":
                        tt = torch.arange(X.shape[-1], device=dev) / FS
                        drift = float(level) * torch.sin(2*math.pi*0.30*tt)[None, None, :]
                        X = X + drift
                    elif corruption == "channel_dropout":
                        X[:, ch.index(channel), :] = 0
                    out = model(X)["wave"].float().cpu().numpy()[:, 0]
                    Yn = Y.numpy()[:, 0]
                    # Compare every model on the paper's [0,1] target scale. WindowDataset
                    # always yields the corpus-native [-1,1] target; NB03's wrapper already
                    # converts predictions, whereas NB04 predictions still need conversion.
                    Yn = (Yn + 1.0) * 0.5
                    if not target_01:
                        out = (out + 1.0) * 0.5
                    for a, bb in zip(Yn, out):
                        mets.append(seg_metrics(a, bb, FS))
            row = {"variant": v, "corruption": corruption, "level": level,
                   "channel": channel, "n": len(mets), "comparison_scale": "[0,1]"}
            for metric in mets[0] if mets else []:
                vals = np.asarray([m[metric] for m in mets], float)
                row[metric] = float(np.nanmean(vals)); row[metric+"_std"] = float(np.nanstd(vals))
            rob.append(row); finished.add(key)
            pd.DataFrame(rob).to_csv(rob_path, index=False)
            sync.mark_dirty(f"robustness:{v}:{corruption}:{level}:{channel}")
            print(f"    {corruption:<16} {str(level):>5} {channel:<8} "
                  f"CC_t {row.get('CC_temporal', float('nan')):6.2f}")
        del model
        gc.collect()
        if dev.type == "cuda":
            torch.cuda.empty_cache()

    ROB = pd.DataFrame(rob)
    if len(ROB):
        ROB.to_csv(WORK / "tables" / "table9_robustness.csv", index=False)
        fig, ax = plt.subplots(figsize=(7.4, 3.8))
        for v in ROB["variant"].unique():
            d = ROB[(ROB["variant"] == v) & (ROB["corruption"] == "awgn")].copy()
            d["level"] = pd.to_numeric(d["level"]); d = d.sort_values("level", ascending=False)
            ax.plot(d["level"], d["CC_temporal"], "o-", lw=1.6, ms=5,
                    color=S["ecg"] if v == "L9_full" else S["muted"],
                    label=NICE.get(v, LAB6.get(v, v)))
        ax.invert_xaxis()
        ax.set_xlabel("input SNR (dB) — noisier to the right")
        ax.set_ylabel("CC_temporal (x100)")
        ax.set_title("Experiment E — graceful degradation under input noise", loc="left")
        ax.legend(frameon=False)
        save(fig, "fig09_robustness.png")
MAJOR("05_robustness")

device: cuda | gpus: 2


Fetching 271 files:   0%|          | 0/271 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  [push_ok] n=3 commit=f7dc126fa837e8069ae096e6b78d72ad7e4f13da msg=nb05_evaluation_v2 major-stage @ 13:30Z
  downloading one selected checkpoint: B_rva__L9_full__f0/best.pt


runs/B_rva__L9_full__f0/best.pt:   0%|          | 0.00/49.8M [00:00<?, ?B/s]


  L9_full: 800 test windows, 8 channel(s)
  [dirty] reason=robustness:L9_full:clean:0:
    clean                0          CC_t  63.01
  [dirty] reason=robustness:L9_full:awgn:12:
    awgn                12          CC_t  60.52
  [dirty] reason=robustness:L9_full:awgn:6:
    awgn                 6          CC_t  55.12
  [dirty] reason=robustness:L9_full:awgn:3:
    awgn                 3          CC_t  49.22
  [dirty] reason=robustness:L9_full:awgn:0:
    awgn                 0          CC_t  40.83
  [dirty] reason=robustness:L9_full:awgn:-3:
    awgn                -3          CC_t  29.45
  [dirty] reason=robustness:L9_full:motion_drift:0.25:
    motion_drift      0.25          CC_t  62.14
  [dirty] reason=robustness:L9_full:motion_drift:0.5:
    motion_drift       0.5          CC_t  59.02
  [dirty] reason=robustness:L9_full:channel_dropout:1:I
    channel_dropout      1 I        CC_t  63.79
  [dirty] reason=robustness:L9_full:channel_dropout:1:Q
    channel_dropout      1 Q        C

runs/B_rva__multireslinknet__f0/best.pt:   0%|          | 0.00/111M [00:00<?, ?B/s]


  multireslinknet: 800 test windows, 1 channel(s)
  [dirty] reason=robustness:multireslinknet:clean:0:
    clean                0          CC_t  57.49
  [dirty] reason=robustness:multireslinknet:awgn:12:
    awgn                12          CC_t   8.92
  [dirty] reason=robustness:multireslinknet:awgn:6:
    awgn                 6          CC_t   3.69
  [dirty] reason=robustness:multireslinknet:awgn:3:
    awgn                 3          CC_t   2.00
  [dirty] reason=robustness:multireslinknet:awgn:0:
    awgn                 0          CC_t   1.55
  [dirty] reason=robustness:multireslinknet:awgn:-3:
    awgn                -3          CC_t   1.15
  [dirty] reason=robustness:multireslinknet:motion_drift:0.25:
    motion_drift      0.25          CC_t  54.68
  [dirty] reason=robustness:multireslinknet:motion_drift:0.5:
    motion_drift       0.5          CC_t  47.55
  [dirty] reason=robustness:multireslinknet:channel_dropout:1:dy
    channel_dropout      1 dy       CC_t  -0.02
  wrote fig0

---
# 8 · Manuscript-ready summary and final push

A single `RESULTS.md` collecting every table, so drafting Section 6 of the paper is transcription
rather than archaeology.

In [10]:
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
L = []
A = L.append
A(f"# CardioMamba-Net — results\n\nGenerated {now} from `{CFG['BASELINE_REPO']}` and `{CFG['MODEL_REPO']}`.\n")
A(f"- runs analysed: **{len(R)}**")
A(f"- experiments: {sorted(R['experiment'].unique())}")
A(f"- variants: {sorted(R['variant'].unique())}\n")
A(f"- input completeness: **{EVALUATION_INPUTS_COMPLETE}**")
A(f"- baseline reproduction gate passed: **{bool(BASELINE_GATE.get('passed', False))}**")
A(f"- evaluation validated: **{EVALUATION_VALIDATED}**\n")

if T3 is not None and len(T3):
    A("## Table 3 — RVA combined (headline)\n")
    A(T3.round(5).to_markdown(index=False)); A("")
if T2 is not None and len(T2):
    A("## Table 2 — per scenario\n"); A(T2.round(5).to_markdown(index=False)); A("")
if TC is not None and len(TC):
    A("## Table 3b — all five scenarios (new)\n"); A(TC.round(5).to_markdown(index=False)); A("")
if TD is not None and len(TD):
    A("## Table 3c — leave-one-subject-out\n"); A(TD.round(5).to_markdown(index=False)); A("")
if TF is not None and len(TF):
    A("## Table 3d — held-out scenario\n"); A(TF.round(5).to_markdown(index=False)); A("")
try:
    A("## Table 4 — R-peak detection\n"); A(T4.to_markdown()); A("")
    A("## Table 5 — HR and HRV (real ms)\n"); A(T5.to_markdown(index=False)); A("")
except Exception:
    pass
if len(T6):
    A("## Table 6 — ablation ladder\n"); A(T6.round(5).to_markdown()); A("")
if len(T7):
    A("## Table 7 — subject-paired Wilcoxon, Holm-corrected\n"); A(T7.to_markdown(index=False)); A("")
if len(T8):
    A("## Table 8 — budget\n"); A(T8[["M_params","CC_temporal","CC_per_Mparam"]].to_markdown()); A("")
if ROBUSTNESS_EFFECTIVE and len(ROB):
    A("## Table 9 — robustness\n"); A(ROB.to_markdown(index=False)); A("")

A("## Figures\n")
for p in sorted(FIG.glob("*.png")):
    A(f"- `figures/{p.name}`")
A("\n## Reading notes for the manuscript\n")
A("- Our splits are strictly subject-wise with non-overlapping test windows. The baseline's "
  "Table 1 counts carry the 50 % overlap and are split 80/20, which permits overlapping windows "
  "across train and test. Baseline rows landing below their published values is the expected "
  "consequence of removing that, not a weaker implementation.")
A("- Correlations are reported x100 throughout, matching the baseline's tables.")
A("- MAE and MSE are standardized to the paper's [0,1] waveform scale. The lossless "
  "all_runs.csv and per-window data retain MAE_recorded/MSE_recorded and the original scale.")
A("- Temporal RRMSE is retained exactly as recorded. Because it depends on the target's DC "
  "offset, it must not be compared directly between NB03 [0,1] and NB04 [-1,1] runs.")
A("- mu_RR is in genuine milliseconds. The baseline's Table 5 reports 126 ms alongside 62 bpm, "
  "which is arithmetically impossible; 126 samples at 128 Hz is 0.98 s.")
A("- Significance uses subject-paired Wilcoxon tests for predeclared full-model comparisons "
  "with Holm correction.")
(WORK / "RESULTS.md").write_text("\n".join(L))

push_state = ("validated-complete" if EVALUATION_VALIDATED else
              "inputs-complete-gate-failed" if EVALUATION_INPUTS_COMPLETE else "partial")
artifact_paths = sorted(
    [p for p in (WORK / "tables").glob("*.csv")] +
    [p for p in (WORK / "figures").glob("*.png")] +
    [WORK / "RESULTS.md", WORK / "input_audit.json"])
artifact_paths = [p for p in artifact_paths if p.exists()]
generation_basis = R.sort_values("run_id").to_json(orient="records", double_precision=12)
generation_id = hashlib.sha256(generation_basis.encode("utf-8")).hexdigest()[:16]
generation = {
    "schema_version": 1,
    "generation_id": generation_id,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "status": push_state,
    "source_repositories": [CFG["BASELINE_REPO"], CFG["MODEL_REPO"]],
    "run_count": int(len(R)),
    "run_ids": sorted(map(str, R["run_id"].tolist())),
    "input_audit": INPUT_AUDIT,
    "artifacts": {
        str(p.relative_to(WORK)): {
            "bytes": int(p.stat().st_size),
            "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        } for p in artifact_paths
    },
}
(WORK / "results_generation.json").write_text(
    json.dumps(generation, indent=2), encoding="utf-8")
sync.mark_dirty(f"canonical-results-rebuilt:{generation_id}")
# `force=True` matters here: major-cell hooks may already have cleared the dirty flag.
# This final commit overwrites RESULTS.md, tables, figures, audit and generation manifest on HF.
ok = sync.flush(final=True, force=True,
                msg=f"{CFG['RUN_ID']} — {push_state}, {len(R)} runs, generation {generation_id}")
sizes = {str(p.relative_to(WORK)): p.stat().st_size for p in WORK.rglob("*") if p.is_file()}
print("\n" + "=" * 76)
if not EVALUATION_INPUTS_COMPLETE:
    status = "PARTIAL EVALUATION — NB04 FULL RUNS MISSING"
elif not EVALUATION_VALIDATED:
    status = "EVALUATION COMPLETE — BASELINE GATE NOT VALIDATED"
else:
    status = "EVALUATION COMPLETE AND VALIDATED"
print(f"  {status}" if ok else f"  {status} (final push had a problem)")
print("=" * 76)
print(f"  repo    : {sync.url}")
print(f"  runs    : {len(R)}")
print(f"  version : {generation_id}")
print(f"  tables  : {len(list((WORK/'tables').glob('*.csv')))}")
print(f"  figures : {len(list(FIG.glob('*.png')))}")
print(f"  payload : {sum(sizes.values())/2**20:.1f} MB")
print("=" * 76)
if EVALUATION_VALIDATED:
    print("\n  RESULTS.md holds every validated table, ready for the manuscript.")
else:
    print("\n  RESULTS.md is an interim report. Do not present it as final/validated yet.")

  [dirty] reason=canonical-results-rebuilt:ca226baa6c1f1ae1


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  [push_ok] n=4 commit=7cbac619945262ae3b463bd7107d6c9482078f80 msg=nb05_evaluation_v2 major-stage @ 13:31Z


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  [push_ok] n=5 commit=bb25560143d1044dc8671ef487160cc0049f7282 msg=nb05_evaluation_v2 — inputs-complete-gate-failed, 180 runs, generation ca226baa6c1f1ae1

  EVALUATION COMPLETE — BASELINE GATE NOT VALIDATED
  repo    : https://huggingface.co/Shanmuk4622/cardiomamba-results-v2
  runs    : 180
  version : ca226baa6c1f1ae1
  tables  : 12
  figures : 9
  payload : 1.1 MB

  RESULTS.md is an interim report. Do not present it as final/validated yet.


---
# 9 · Troubleshooting

**`No runs found`** — NB03 and NB04 push to separate `BASELINE_REPO` and `MODEL_REPO`.
Check both names and that at least one run finished.

**Table 7 skipped** — expected while only quick results exist. Finish the canonical NB04 queue
(`QUICK = False`) so every held-out subject has paired results.

**`PARTIAL EVALUATION — NB04 FULL RUNS MISSING`** — the notebook is working correctly, but only
quick or incomplete NB04 runs exist. Quick runs are excluded from paper tables. Finish NB04 and
re-run NB05; all existing baseline tables remain resumable in the results repository.

**Bland–Altman plots empty** — the per-subject metrics come from `metrics_subjects.parquet`, which
is only written when a test split has at least two windows per subject. Re-run with the full folds.

**Robustness section slow or out of memory** — lower `CFG["ROBUST_MAX_WINDOWS"]`, or set
`CFG["RUN_ROBUSTNESS"] = False` and run it in its own session.

**A checkpoint fails to load** — architecture config drifted between training and evaluation. The
loader uses `strict=True` and stops on any mismatch. If you changed `CFG["BASE"]` or `D_SSM`
after training, restore the recorded configuration rather than forcing a partial load.